# SPACECONS — consensus of 5 spatial trajectory methods (local-server version)

Runs on a Linux/macOS server (a GPU helps for SpaceFlow and CASCAT but is not required). **Methods:** SpaceFlow, SpaTrack, StaVia, stLearn PSTS, CASCAT — each keeps its own milestones and its own graph.
**SPACECONS** (sections 8–10): consensus pseudotime (**R**), consensus milestones and milestone graph (**M**), and validation against a reference trajectory with CASCAT's metrics.

**Data:** figshare article 28342754, version 3 (downloaded automatically, or point `LOCAL_DATA` at a copy).

## How to run
1. **Edit the settings cell** (next cell). At minimum, check `FIGSHARE_*`/`LOCAL_DATA` and `SPECIES`.
2. Run top to bottom. The *data cell* lists every file it finds and every column of the `.h5ad` — use that to fill in `REF_MODE`, `ANNOT_COLUMN`, `STATE_MAP`, `ROOT_SPOT` … and run from the *load* cell again.
3. First run needs internet (pip, figshare, GitHub, the helper environments ≈ 10–15 min). Afterwards everything is cached below `WORK_DIR`.
4. Headless: `jupyter nbconvert --to notebook --execute --ExecutePreprocessor.timeout=-1 SPACECONS_local.ipynb`

## What changed compared with the Kaggle notebook
* every `/kaggle/...` path and the `/tmp` folders → `WORK_DIR` (settings cell); `!pip` → pip of the running kernel; `uv` is called as `python -m uv`;
* the TBI / breast-cancer download and hard-coded hypothalamus / DCIS logic → generic loader (10x Visium folder **or** `.h5ad`) and a configurable reference (`REF_MODE`);
* helper environments (stLearn, CASCAT) can use CPU or another CUDA build; stopping a stuck CASCAT run kills only its own process group (the old `pkill -f` could hit other users' jobs on a shared server);
* data without an H&E image works (blank canvas for the maps; stLearn runs without morphology);
* the "second dataset" machinery (re-running cells through IPython history) is removed — one run = one dataset/sample.

In [ ]:
# =====================================  SETTINGS — edit this cell, then run the notebook top to bottom  =====================================
import os, sys, glob, json, time, shutil, subprocess, tarfile, zipfile, gzip, re

# ---------- base directory (relative to the folder where Jupyter started) ----------
if '_NOTEBOOK_DIR' not in globals():
    _NOTEBOOK_DIR = os.getcwd()                       # remembered once so re-running this cell never shifts paths

BASE_REL_DIR = '.'                                    # relative path to your base directory (e.g. '.', '..', '../project')
BASE_DIR     = os.path.abspath(os.path.join(_NOTEBOOK_DIR, BASE_REL_DIR))

def from_base(path):
    """Resolve a path relative to BASE_DIR (returns None if path is None; keeps absolute paths unchanged)."""
    if path is None:
        return None
    return path if os.path.isabs(path) else os.path.abspath(os.path.join(BASE_DIR, path))

WORK_DIR = from_base('spacecons_work')                # data, helper environments and results are all written below this folder
RUN_NAME = 'OSCC2'                                    # results go to <WORK_DIR>/results_<RUN_NAME>   (change it when you run another sample)

# ---------- data ----------
FIGSHARE_ID, FIGSHARE_VERSION = 28342754, 3
LOCAL_DATA   = from_base('OSCC2.zip')                 # relative to BASE_DIR: zip/tar archive, folder (e.g. 'OSCC2'), or .h5ad file. None = download from figshare
INPUT_PATH   = from_base(None)                        # optional: force ONE Visium folder or .h5ad file (relative to BASE_DIR)
SAMPLE_HINT  = None                                   # several samples in the download: a part of the wanted sample's path, e.g. 'OSCC2'
SAMPLE_COLUMN, SAMPLE_VALUE = None, None              # ONE .h5ad holding several samples: the obs column and the value to keep
COORD_COLS   = None                                   # .h5ad without obsm['spatial'] AND without a sibling spatial/ folder: ('x', 'y')
COUNTS_LAYER = None                                   # .h5ad whose .X is not raw counts: name of the layer with raw counts (None = use .X, or .raw when .X is normalised)
VISIUM_COUNT_FILE = 'filtered_feature_bc_matrix.h5'
SPECIES      = 'Human'                                # 'Human' or 'Mouse' (passed to CASCAT)
MIN_GENES, MIN_CELLS = 200, 3                         # QC: drop spots with fewer detected genes / genes seen in fewer spots
LEIDEN_RES   = 0.8                                    # resolution of the expression regions (the shared "regions" used by the comparison / consensus)

# ---------- reference trajectory: ONLY used to choose the root and to validate (section 10); never used to build the results ----------
#   'none'       : no reference. Root = ROOT_SPOT, or ROOT_REGION, or (last resort) a CytoTRACE-like guess. Validation is skipped.
#   'annotation' : every spot has a label (a column of the .h5ad's obs, or ANNOT_FILE); STATE_MAP turns labels into the ordered states of REF_STATES.
#   'markers'    : states are recognised from marker genes (as in the TBI brain example). MARKERS = {state: [genes]} in trajectory order.
REF_MODE     = 'none'
ANNOT_FILE   = from_base(None)                        # csv/tsv relative to BASE_DIR (first column = spot barcode). None = take ANNOT_COLUMN from the .h5ad's obs
ANNOT_COLUMN = 'cluster'                              # e.g. 'cluster' or 'pathologist_anno.x' in OSCC2/data.h5ad
REF_STATES   = ['early_state', 'late_state']          # the stated trajectory, earliest -> latest (annotation mode)
STATE_MAP    = {'label_in_data_1': 'early_state', 'label_in_data_2': 'late_state'}   # data label -> state (labels not listed are ignored; keys as text)
MARKERS      = {}                                     # markers mode, e.g. {'hypothalamus': ['Avp', 'Oxt'], 'thalamus': [...], 'hippocampus': [...], 'injury': [...]}
ROOT_SPOT    = None                                   # start the trajectory at exactly this spot barcode, or ...
ROOT_REGION  = None                                   # ... in this expression region (a Leiden label such as '3', printed by the root cell); only used when REF_MODE = 'none'

# ---------- compute ----------
INSTALL_MAIN = True                                   # pip-install the main-environment packages (set False once done)
N_THREADS    = min(8, os.cpu_count() or 4)
CASCAT_CUDA  = 'auto'                                 # 'auto' | 'cu118' | 'cu121' | 'cpu' — CUDA build of CASCAT's own environment ('auto' = cu118 if nvidia-smi works, else cpu)
CASCAT_YAML  = 'HER2ST.yml'                           # CASCAT settings file (the authors' Visium settings)
CASCAT_TRIALS, CASCAT_EPOCHS = 2, 50                  # authors' default is 5 x 100 (training is quick; the slow part is the causal tree)
CASCAT_KDE   = 0                                      # 0 = CASCAT's binned CMI estimator in the tree step (minutes); 1 = authors' kernel-density CMI (hours)
CASCAT_MAX_MIN = 60                                   # CASCAT is stopped and skipped after this many minutes so that the notebook continues
STL_MAX_MIN  = 120                                    # same for stLearn

# ---------- derived paths (no need to edit) ----------
DATA_DIR = f'{WORK_DIR}/data'; ENV_DIR = f'/home/user/Gagan_env/genomicsProject/envs'; OUT = f'{WORK_DIR}/results_{RUN_NAME}'
CASCAT_WORK = f'{WORK_DIR}/cascat_run'; CASCAT_RESULT = f'{CASCAT_WORK}/result/data_processed.h5ad'; STL_TILES = f'{WORK_DIR}/stl_tiles'
STL_PY = f'{ENV_DIR}/stl/bin/python'; CAS_PY = f'{ENV_DIR}/cascat/bin/python'; CASCAT_DIR = f'{ENV_DIR}/CASCAT'
for d_ in (DATA_DIR, ENV_DIR, OUT): os.makedirs(d_, exist_ok=True)
os.chdir(WORK_DIR)                                    # helper scripts written by this notebook (run_stlearn_psts.py, run_cascat.py) are saved here
if CASCAT_CUDA == 'auto':
    CASCAT_CUDA = 'cu118' if shutil.which('nvidia-smi') and subprocess.run(['nvidia-smi'], capture_output=True).returncode == 0 else 'cpu'
CASCAT_DEVICE = 'cpu' if CASCAT_CUDA == 'cpu' else 'cuda'
CASCAT_DEVICE = 'cpu'
print('base folder:', BASE_DIR, '\nwork folder:', WORK_DIR, '\nlocal data :', LOCAL_DATA, '\nresults    :', OUT, '\nCASCAT build:', CASCAT_CUDA)

In [ ]:
# ---- small helpers used by the later cells ----
import threading, queue, signal
import numpy as np, pandas as pd
from scipy.stats import rankdata, spearmanr as _spearmanr
_trapz = getattr(np, 'trapezoid', None) or getattr(np, 'trapz')          # np.trapz was renamed in NumPy 2

def sh(cmd):
    r = subprocess.run(cmd, shell=True, capture_output=True, text=True)
    print(('OK   ' if r.returncode == 0 else 'FAIL ') + cmd[:130])
    if r.returncode: print(r.stderr[-1500:])
    return r.returncode == 0

def run_live(cmd, skip=lambda l: False, timeout_min=None, done_file=None):
    """Run a separate program and show its output live, with elapsed time. Stops it at the time limit, or 2 min after `done_file`
    appeared with no new output (avoids an end-of-run hang). Only the program's own process group is killed."""
    t0 = time.time(); env = {**os.environ, 'MPLBACKEND': 'Agg', 'PYTHONUNBUFFERED': '1'}
    p = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, env=env, bufsize=1, start_new_session=True)
    def kill():
        try: os.killpg(os.getpgid(p.pid), signal.SIGKILL)
        except Exception: p.kill()
    q = queue.Queue(); threading.Thread(target=lambda: [q.put(l) for l in p.stdout], daemon=True).start()
    tail, last = [], time.time()
    while True:
        try:
            line = q.get(timeout=5).rstrip(); last = time.time(); tail = (tail + [line])[-40:]
            if line and not skip(line): print(f'[{(time.time() - t0) / 60:5.1f} min] {line[:200]}', flush=True)
            continue
        except queue.Empty: pass
        if p.poll() is not None: break
        if timeout_min and time.time() - t0 > timeout_min * 60:
            kill(); print(f'--- stopped: time limit of {timeout_min} min reached'); return -9, '\n'.join(tail)
        if done_file and os.path.exists(done_file) and time.time() - last > 120:
            kill(); print('--- result saved but the process did not exit; stopped it'); return 0, '\n'.join(tail)
    while not q.empty():                                   # lines that arrived just before the program ended
        line = q.get().rstrip(); tail = (tail + [line])[-40:]
        if line and not skip(line): print(f'[{(time.time() - t0) / 60:5.1f} min] {line[:200]}', flush=True)
    print(f'--- finished in {(time.time() - t0) / 60:.1f} min, exit code {p.returncode}')
    return p.returncode, '\n'.join(tail)

def spearmanr(a, b, **k):                     # ignores spots without a value (e.g. spots outside the reference states); NaN when nothing is left
    a, b = np.asarray(a, float), np.asarray(b, float)
    if np.isfinite(a).sum() < 3 or np.isfinite(b).sum() < 3: return (np.nan, np.nan)
    return _spearmanr(a, b, nan_policy='omit')

def rank01(t):
    t = np.asarray(t, dtype=float)
    t = np.where(np.isfinite(t), t, np.nanmax(t[np.isfinite(t)]))
    return (rankdata(t) - 1) / (len(t) - 1)

def get_he(ad_, xy_, lib_hint=None):
    """H&E image + spot positions on it. Without an image in the data: a blank canvas that fits the spots (maps still work)."""
    sp_ = ad_.uns.get('spatial', {}) or {}
    if sp_:
        lib = lib_hint if lib_hint in sp_ else list(sp_.keys())[0]
        imgs, sfs = sp_[lib].get('images', {}), sp_[lib].get('scalefactors', {})
        for q_ in ('hires', 'lowres'):
            if q_ in imgs and f'tissue_{q_}_scalef' in sfs:
                return np.asarray(imgs[q_]), xy_ * sfs[f'tissue_{q_}_scalef'], True
    pad = 0.03 * float((xy_.max(0) - xy_.min(0)).max()); p_ = xy_ - (xy_.min(0) - pad)
    W, H = np.ceil(p_.max(0) + pad).astype(int)
    return np.ones((H, W, 3)), p_, False

In [ ]:
# Main environment = the Python that runs this notebook. Use a fresh venv / conda env with Python 3.10–3.12 (and NumPy 2).
# 'dependency conflict' messages are expected and harmless. For a specific CUDA build of PyTorch, install it yourself first, e.g.
#   pip install torch --index-url https://download.pytorch.org/whl/cu121
if INSTALL_MAIN:
    PIP = f'"{sys.executable}" -m pip install -q'
    sh(f'{PIP} --no-deps SpaceFlow spaTrack')
    sh(f'{PIP} torch torch-geometric gudhi cmcrameri scanpy igraph leidenalg pot pygam numpy pandas scipy scikit-learn matplotlib networkx h5py requests pyyaml tqdm plotly')
    sh(f'{PIP} --no-deps pyVIA pecanpy nptyping numba-progress s_gd2 graphtools tasklogger pygsp phate scprep')
    sh(f'{PIP} hnswlib datashader dask scikit-image "gensim>=4.4" deprecated future termcolor gdown wget umap-learn seaborn uv')
else:
    print('INSTALL_MAIN = False -> skipped')

## Separate environments for stLearn and CASCAT
stLearn only installs on Python < 3.13 with NumPy < 2, and CASCAT's authors pin Python 3.10 + PyTorch 2.1.1 + PyG 2.6.1. Each therefore gets its **own small environment** (made with `uv`, which downloads the Python version itself; `git` is needed for CASCAT).
They run as separate programs, read the same data and hand back one pseudotime value per spot. First time ≈ 10 minutes; afterwards the cell only re-checks that they exist.

In [ ]:
# Two separate environments, made with `uv`:  stLearn -> Python 3.11 / NumPy < 2;   CASCAT -> Python 3.10, PyTorch 2.1.1 and PyG 2.6.1 exactly as in its README
sh(f'"{sys.executable}" -m pip install -q uv')
UV = f'"{sys.executable}" -m uv'
TORCH_INDEX = f'https://download.pytorch.org/whl/{CASCAT_CUDA}'
PYG_INDEX = f'https://data.pyg.org/whl/torch-2.1.0+{CASCAT_CUDA}.html'
CUPY = {'cu118': 'cupy-cuda11x', 'cu121': 'cupy-cuda12x'}.get(CASCAT_CUDA, '')        # cupy: GPU speed-up recommended in the CASCAT README
if not os.path.exists(STL_PY):
    sh(f'{UV} venv -q -p 3.11 "{ENV_DIR}/stl"')
    sh(f'{UV} pip install -q -p "{STL_PY}" stlearn==1.2.2 "setuptools<81"')
if not os.path.exists(CAS_PY):
    if not os.path.exists(CASCAT_DIR): sh(f'git clone -q --depth 1 https://github.com/deepomicslab/CASCAT "{CASCAT_DIR}"')
    sh(f'{UV} venv -q -p 3.10 "{ENV_DIR}/cascat"')
    sh(f'{UV} pip install -q -p "{CAS_PY}" torch==2.1.1 --index-url {TORCH_INDEX}')
    sh(f'{UV} pip install -q -p "{CAS_PY}" torch_geometric==2.6.1 pyg_lib torch_scatter torch_sparse torch_cluster torch_spline_conv -f {PYG_INDEX}')
    sh(f'{UV} pip install -q -p "{CAS_PY}" scanpy==1.10.1 matplotlib networkx scikit-misc pydot pot numpy==1.26.4 scikit-learn==1.5.2 '
       f'numba==0.60.0 numba-scipy==0.4.0 pandas==2.2.3 scipy==1.11.0 pyyaml==6.0.3 umap-learn seaborn {CUPY}')
sh(f'MPLBACKEND=Agg "{STL_PY}" -c "import stlearn, numpy; print(stlearn.__version__, numpy.__version__)"')
sh(f'MPLBACKEND=Agg "{CAS_PY}" -c "import torch, torch_sparse, torch_geometric; print(torch.__version__, torch.cuda.is_available())"')

## Get the data and find out what is in it
Downloads figshare article `FIGSHARE_ID` (version `FIGSHARE_VERSION`) with the figshare API (md5-checked), unpacks any zip/tar files, **lists every file**, and looks for
* a **10x Visium folder** (`filtered_feature_bc_matrix.h5` + `spatial/`), or
* an **`.h5ad`** file (it prints its obs columns, so you can fill in `ANNOT_COLUMN`, `SAMPLE_COLUMN`, `COORD_COLS`).

If the server has no internet access (or figshare blocks the download), download the files in a browser, copy them to the server and set `LOCAL_DATA` to that folder.

In [ ]:
# [data]  download from figshare (or use LOCAL_DATA), unpack, list, detect the input
import requests, hashlib
os.makedirs(DATA_DIR, exist_ok=True)

def _md5(path):
    h = hashlib.md5()
    with open(path, 'rb') as f:
        for chunk in iter(lambda: f.read(1 << 20), b''): h.update(chunk)
    return h.hexdigest()

def _is_junk(path):
    parts = os.path.normpath(path).split(os.sep)
    base = parts[-1]
    return ('__MACOSX' in parts) or base.startswith('._') or base == '.DS_Store'

def download(url, dest, size=None, md5=None):
    if os.path.exists(dest) and (size is None or os.path.getsize(dest) == size):
        print('  already downloaded:', os.path.basename(dest)); return
    tmp = dest + '.part'
    with requests.get(url, stream=True, timeout=120, headers={'User-Agent': 'Mozilla/5.0'}) as r:
        r.raise_for_status()
        with open(tmp, 'wb') as f:
            for chunk in r.iter_content(1 << 20): f.write(chunk)
    os.replace(tmp, dest)
    if md5: print(('  md5 ok: ' if _md5(dest) == md5 else '  !! MD5 MISMATCH (damaged download?): ') + os.path.basename(dest))

def _extract_archive(archive_path, dest):
    low = archive_path.lower()
    if os.path.exists(dest):
        return False
    print('  unpacking', os.path.basename(archive_path), '->', dest)
    if low.endswith(('.h5ad.gz', '.h5.gz')):
        with gzip.open(archive_path, 'rb') as a_, open(dest, 'wb') as b_: shutil.copyfileobj(a_, b_)
    elif low.endswith('.zip'):
        with zipfile.ZipFile(archive_path) as z_: z_.extractall(dest)
    else:
        with tarfile.open(archive_path) as t_:
            try: t_.extractall(dest, filter='data')
            except TypeError: t_.extractall(dest)
    return True

def unpack_all(root):                    # unpack zip / tar(.gz) / .h5ad.gz (also when nested); every archive is unpacked once into <name>_x
    changed = True
    while changed:
        changed = False
        for f in sorted(glob.glob(f'{root}/**/*', recursive=True)):
            if not os.path.isfile(f) or _is_junk(f): continue
            low = f.lower()
            if low.endswith('.zip'):
                dest = re.sub(r'\.zip$', '', f, flags=re.I) + '_x'
            elif low.endswith(('.tar.gz', '.tgz', '.tar')):
                dest = re.sub(r'\.(tar\.gz|tgz|tar)$', '', f, flags=re.I) + '_x'
            elif low.endswith(('.h5ad.gz', '.h5.gz')):
                dest = f[:-3]
            else:
                continue
            if _extract_archive(f, dest):
                changed = True

if LOCAL_DATA:
    if not os.path.exists(LOCAL_DATA):
        raise FileNotFoundError(f'LOCAL_DATA not found: {LOCAL_DATA}  (BASE_DIR = {BASE_DIR})')
    low_ld = LOCAL_DATA.lower()
    if os.path.isfile(LOCAL_DATA) and low_ld.endswith(('.zip', '.tar.gz', '.tgz', '.tar')):
        stem = re.sub(r'\.(tar\.gz|tgz|tar|zip)$', '', os.path.basename(LOCAL_DATA), flags=re.I)
        SRC_ROOT = os.path.join(DATA_DIR, stem + '_x')
        _extract_archive(LOCAL_DATA, SRC_ROOT)
    else:
        SRC_ROOT = LOCAL_DATA
else:
    SRC_ROOT = f'{DATA_DIR}/figshare_{FIGSHARE_ID}_v{FIGSHARE_VERSION}'; os.makedirs(SRC_ROOT, exist_ok=True)
    try:
        meta = requests.get(f'https://api.figshare.com/v2/articles/{FIGSHARE_ID}/versions/{FIGSHARE_VERSION}', timeout=60, headers={'User-Agent': 'Mozilla/5.0'})
        meta.raise_for_status(); meta = meta.json()
        print('figshare:', meta.get('title'), '| DOI', meta.get('doi'), '|', len(meta['files']), 'files')
        for f in meta['files']:
            if f.get('is_link_only'): print('  (link only, not downloaded)', f['name'], f['download_url']); continue
            print(f"  {f['name']}  ({f['size'] / 1e6:.1f} MB)")
            download(f['download_url'], os.path.join(SRC_ROOT, f['name']), f['size'], f.get('computed_md5') or f.get('supplied_md5'))
    except Exception as e:
        print('figshare API failed (', repr(e)[:160], ') -> trying the whole-article zip')
        try: download(f'https://ndownloader.figshare.com/articles/{FIGSHARE_ID}/versions/{FIGSHARE_VERSION}', f'{SRC_ROOT}/article.zip')
        except Exception as e2:
            raise RuntimeError('Could not download from figshare (' + repr(e2)[:160] + '). Download the files in a browser, copy them to this server '
                               'and set LOCAL_DATA in the settings cell to that folder.')

if os.path.isdir(SRC_ROOT): unpack_all(SRC_ROOT)

files = sorted(f for f in glob.glob(f'{SRC_ROOT}/**/*', recursive=True) if os.path.isfile(f) and not _is_junk(f)) if os.path.isdir(SRC_ROOT) else [SRC_ROOT]
print(f'\nFiles ({len(files)}):')
for f in files[:200]: print(f'  {os.path.relpath(f, SRC_ROOT) if os.path.isdir(SRC_ROOT) else f}  ({os.path.getsize(f) / 1e6:.1f} MB)')
if len(files) > 200: print('  ...')

visium_dirs = [os.path.dirname(p) for p in glob.glob(f'{SRC_ROOT}/**/{VISIUM_COUNT_FILE}', recursive=True)
               if not _is_junk(p) and os.path.isdir(os.path.join(os.path.dirname(p), 'spatial'))]
h5ads = [f for f in files if f.lower().endswith('.h5ad')]
if SAMPLE_HINT:
    visium_dirs = [d for d in visium_dirs if SAMPLE_HINT.lower() in d.lower()]
    h5ads = [f for f in h5ads if SAMPLE_HINT.lower() in f.lower()]
print('\nVisium folders found:', visium_dirs or 'none')
print('.h5ad files found   :', h5ads or 'none')

VISIUM_DIR = H5AD_PATH = SPATIAL_DIR = None
if INPUT_PATH:
    p_ = INPUT_PATH; INPUT_KIND = 'h5ad' if p_.lower().endswith('.h5ad') else 'visium'
elif visium_dirs: INPUT_KIND, p_ = 'visium', visium_dirs[0]
elif h5ads:       INPUT_KIND, p_ = 'h5ad', h5ads[0]
else:
    raise FileNotFoundError('Neither a Visium folder (with ' + VISIUM_COUNT_FILE + ' and spatial/) nor an .h5ad file was found - see the file list above. '
                            'Other formats (csv count matrix, 10x mtx folder, Stereo-seq, ...) need a small conversion to .h5ad first.')

if INPUT_KIND == 'visium':
    VISIUM_DIR = p_
else:
    H5AD_PATH = p_
    sib_sp = os.path.join(os.path.dirname(H5AD_PATH), 'spatial')
    if os.path.isdir(sib_sp):
        SPATIAL_DIR = sib_sp

if (len(visium_dirs) + len(h5ads)) > 1 and not INPUT_PATH:
    print('\n!! more than one candidate: using the first. Set SAMPLE_HINT or INPUT_PATH in the settings cell to choose another one.')
print(f'\n=> input: {INPUT_KIND} -> {p_}')

if INPUT_KIND == 'visium':
    print('   folder:', sorted(os.listdir(VISIUM_DIR)), '| spatial/:', sorted(f for f in os.listdir(os.path.join(VISIUM_DIR, 'spatial')) if not _is_junk(f)))
else:
    import scanpy as sc
    if SPATIAL_DIR:
        print('   sibling spatial/:', sorted(f for f in os.listdir(SPATIAL_DIR) if not _is_junk(f)))
    a_ = sc.read_h5ad(H5AD_PATH, backed='r')
    print(f'\n.h5ad: {a_.n_obs} spots x {a_.n_vars} genes | obsm: {list(a_.obsm.keys())} | layers: {list(a_.layers.keys())} | raw: {a_.raw is not None} | uns: {list(a_.uns.keys())[:10]}')
    print('obs columns  (-> ANNOT_COLUMN / SAMPLE_COLUMN / COORD_COLS in the settings cell):')
    for c_ in a_.obs.columns:
        s_ = a_.obs[c_]; nu_ = s_.nunique()
        print(f'   {c_:30s} {str(s_.dtype):10s} {nu_:6d} unique' + (' | ' + ', '.join(map(str, s_.value_counts().index[:12])) if nu_ <= 30 else ''))
    a_.file.close()

In [ ]:
# Save EVERY figure the notebook shows as a PNG file too (into <results folder>/figures/), so all of them can be downloaded
import itertools, matplotlib.pyplot as plt
if not hasattr(plt, '_show_original'):
    plt._show_original = plt.show; _FIG_NO = itertools.count(1)
    def _show_and_save(*a, **k):
        os.makedirs(f'{OUT}/figures', exist_ok=True)
        for num in plt.get_fignums():
            plt.figure(num).savefig(f'{OUT}/figures/fig_{next(_FIG_NO):03d}.png', dpi=150, bbox_inches='tight')
        return plt._show_original(*a, **k)
    plt.show = _show_and_save
print('every figure will also be saved to', f'{OUT}/figures/')


## 0. Load the spatial data, QC, preprocess
QC: drop spots with fewer than `MIN_GENES` detected genes and genes seen in fewer than `MIN_CELLS` spots. Then: normalise → log → 2,000 highly variable genes → PCA (30) → neighbour graph → Leiden clusters (the expression **regions**).

In [ ]:
# [load]
import scanpy as sc, anndata, scipy.sparse as sp, matplotlib.pyplot as plt
from PIL import Image

def _attach_sibling_spatial(adata, spatial_dir, lib_id='sample'):
    """Load Visium coordinates, H&E image, and scalefactors from a sibling spatial/ folder into adata.
    Also fixes barcode suffix mismatches (e.g. '-2' vs '-1') and swapped (row, col) obsm['spatial'] coordinates."""
    # 1) Spot coordinates from tissue_positions(.csv|_list.csv)
    pos_file = next(
        (os.path.join(spatial_dir, f) for f in ('tissue_positions.csv', 'tissue_positions_list.csv')
         if os.path.exists(os.path.join(spatial_dir, f))),
        None
    )
    if pos_file:
        first_row = open(pos_file).readline().strip().split(',')
        has_header = len(first_row) >= 6 and first_row[1].strip().lower() == 'in_tissue'
        if has_header:
            pos = pd.read_csv(pos_file, index_col=0)
        else:
            pos = pd.read_csv(
                pos_file, header=None, index_col=0,
                names=['barcode', 'in_tissue', 'array_row', 'array_col', 'pxl_row_in_fullres', 'pxl_col_in_fullres']
            )
        pos.index = pos.index.astype(str)
        pos_aligned = None

        matched = adata.obs_names.isin(pos.index)
        if matched.sum() > 0:
            if matched.sum() < adata.n_obs and 'spatial' not in adata.obsm:
                print(f'  dropping {adata.n_obs - int(matched.sum())} spots missing from {os.path.basename(pos_file)}')
                adata = adata[matched].copy()
                matched = np.ones(adata.n_obs, dtype=bool)
            if matched.all():
                pos_aligned = pos.loc[adata.obs_names]
        else:
            # fallback: match after stripping trailing '-1' / '-2' barcode suffix
            strip = lambda ix: pd.Index(ix).astype(str).str.replace(r'-\d+$', '', regex=True)
            pos_s = pos.copy(); pos_s.index = strip(pos_s.index); pos_s = pos_s[~pos_s.index.duplicated()]
            obs_s = strip(adata.obs_names)
            matched = obs_s.isin(pos_s.index)
            if matched.sum() > 0:
                if matched.sum() < adata.n_obs and 'spatial' not in adata.obsm:
                    print(f'  dropping {adata.n_obs - int(matched.sum())} spots missing from {os.path.basename(pos_file)}')
                    adata = adata[matched].copy()
                    obs_s = obs_s[matched]
                    matched = np.ones(adata.n_obs, dtype=bool)
                if matched.all():
                    pos_aligned = pos_s.loc[obs_s]
                    pos_aligned.index = adata.obs_names

        if pos_aligned is not None and {'pxl_col_in_fullres', 'pxl_row_in_fullres'}.issubset(pos_aligned.columns):
            expected_xy = pos_aligned[['pxl_col_in_fullres', 'pxl_row_in_fullres']].to_numpy(dtype=float)
            if 'spatial' not in adata.obsm:
                adata.obsm['spatial'] = expected_xy
                print(f'  loaded obsm["spatial"] from spatial/{os.path.basename(pos_file)} ({adata.n_obs} spots)')
            else:
                cur_xy = np.asarray(adata.obsm['spatial'], dtype=float)[:, :2]
                if np.allclose(cur_xy, expected_xy[:, [1, 0]], atol=1.0) and not np.allclose(cur_xy, expected_xy, atol=1.0):
                    adata.obsm['spatial'] = expected_xy
                    print('  detected swapped (pxl_row, pxl_col) in obsm["spatial"] -> corrected to standard (pxl_col, pxl_row) to align with H&E')
            for c_ in ('in_tissue', 'array_row', 'array_col'):
                if c_ in pos_aligned.columns and c_ not in adata.obs.columns:
                    adata.obs[c_] = pos_aligned[c_].values

    # 2) H&E image + scalefactors into adata.uns['spatial'] (stored as uint8 RGB for both matplotlib and stLearn)
    has_img = any(len(v_.get('images', {})) > 0 for v_ in (adata.uns.get('spatial') or {}).values())
    if not has_img:
        sfs_file = os.path.join(spatial_dir, 'scalefactors_json.json')
        sfs = json.load(open(sfs_file)) if os.path.exists(sfs_file) else {}
        imgs = {}
        for q_, fn_ in (('hires', 'tissue_hires_image.png'), ('lowres', 'tissue_lowres_image.png')):
            ip_ = os.path.join(spatial_dir, fn_)
            if os.path.exists(ip_):
                imgs[q_] = np.asarray(Image.open(ip_).convert('RGB'), dtype=np.uint8)
        if imgs:
            adata.uns.setdefault('spatial', {})[lib_id] = {
                'images': imgs,
                'scalefactors': sfs,
                'use_quality': 'hires' if 'hires' in imgs else 'lowres',
            }
            print(f'  loaded H&E image ({list(imgs.keys())}) and scalefactors from spatial/')
    return adata

if INPUT_KIND == 'visium':
    adata = sc.read_visium(VISIUM_DIR, count_file=VISIUM_COUNT_FILE)
else:
    adata = sc.read_h5ad(H5AD_PATH)
    if SAMPLE_COLUMN:
        adata = adata[adata.obs[SAMPLE_COLUMN].astype(str) == str(SAMPLE_VALUE)].copy()
        print(f'kept sample {SAMPLE_COLUMN} = {SAMPLE_VALUE}:', adata.shape)
    adata.obs_names_make_unique(); adata.var_names_make_unique()

    if SPATIAL_DIR:
        lib_id = os.path.basename(os.path.dirname(H5AD_PATH)) or RUN_NAME
        adata = _attach_sibling_spatial(adata, SPATIAL_DIR, lib_id=lib_id)

    if 'spatial' not in adata.obsm:
        if not COORD_COLS:
            raise ValueError(
                "The .h5ad has no obsm['spatial'] and no matching spatial/tissue_positions*.csv was found: "
                f"set COORD_COLS = ('x_column', 'y_column') in the settings cell. Available obs columns: {list(adata.obs.columns)}"
            )
        adata.obsm['spatial'] = adata.obs[list(COORD_COLS)].to_numpy(float)

    def _is_counts(X):
        v = X.data if sp.issparse(X) else np.asarray(X).ravel()
        v = np.asarray(v[:500000]); return len(v) > 0 and v.min() >= 0 and np.allclose(v, np.round(v))
    if COUNTS_LAYER:
        adata.X = adata.layers[COUNTS_LAYER].copy()
    elif not _is_counts(adata.X):
        if adata.raw is not None and _is_counts(adata.raw.X):
            print('.X is normalised -> using .raw.X (raw counts)')
            adata = anndata.AnnData(X=adata.raw.X, obs=adata.obs, var=adata.raw.var, obsm=adata.obsm, uns=adata.uns)
        else:
            print('WARNING: .X does not look like raw counts and no raw counts were found (set COUNTS_LAYER). SpaceFlow / stLearn / CASCAT normalise by themselves, so results may be wrong.')

adata.X = adata.X.tocsr() if sp.issparse(adata.X) else sp.csr_matrix(adata.X)
adata.obs_names_make_unique(); adata.var_names_make_unique()
print('raw:', adata.shape)

In [ ]:
adata

In [ ]:
adata.obs['cluster'].nunique()

In [ ]:
sc.pp.filter_cells(adata, min_genes=MIN_GENES)
sc.pp.filter_genes(adata, min_cells=MIN_CELLS)
raw = adata.copy()                                   # raw counts (SpaceFlow / stLearn / CASCAT normalise themselves)
ad = adata.copy()
sc.pp.normalize_total(ad, target_sum=1e4); sc.pp.log1p(ad)
sc.pp.highly_variable_genes(ad, n_top_genes=min(2000, ad.n_vars - 1))
sc.tl.pca(ad, n_comps=30, mask_var='highly_variable', random_state=0)
sc.pp.neighbors(ad, n_neighbors=15, n_pcs=30, random_state=0)
sc.tl.leiden(ad, resolution=LEIDEN_RES, key_added='region', flavor='igraph', n_iterations=2, directed=False, random_state=0)
xy = np.asarray(ad.obsm['spatial'], dtype=float)[:, :2]
open(f'{OUT}/kept_spots.txt', 'w').write('\n'.join(ad.obs_names))
IMG, PXY, HAS_IMAGE = get_he(ad, xy, SAMPLE_VALUE)
print('after QC:', ad.shape, '| expression clusters (regions):', ad.obs['region'].nunique(), '| H&E image in the data:', HAS_IMAGE)

In [ ]:
ad

In [ ]:
def mini_adata(src):                                 # small AnnData for the helper environments: raw counts + coordinates, no extra columns that could break writing
    keep_var = [c for c in ('gene_ids', 'feature_types', 'genome') if c in src.var.columns]
    keep_obs = [c for c in ('in_tissue', 'array_row', 'array_col') if c in src.obs.columns]
    m = anndata.AnnData(X=src.X.copy(), obs=src.obs[keep_obs].copy(), var=src.var[keep_var].copy())
    m.obsm['spatial'] = np.asarray(src.obsm['spatial'], float)
    return m

fig, axes = plt.subplots(1, 2, figsize=(13, 6))
if HAS_IMAGE: axes[0].imshow(IMG); axes[0].set_title('H&E image'); axes[0].axis('off')
else:
    s_ = axes[0].scatter(xy[:, 0], -xy[:, 1], c=np.log1p(ad.obs['n_genes']), s=6); plt.colorbar(s_, ax=axes[0])
    axes[0].set_title('log(genes per spot)  (no image in the data)'); axes[0].set_aspect('equal'); axes[0].axis('off')
for lab_ in ad.obs['region'].cat.categories:
    m = (ad.obs['region'] == lab_).values; axes[1].scatter(xy[m, 0], -xy[m, 1], s=6, label=lab_)
axes[1].legend(fontsize=7, markerscale=2, ncol=2); axes[1].set_aspect('equal'); axes[1].axis('off'); axes[1].set_title('Expression clusters (Leiden)')
plt.tight_layout(); plt.show()

## Root spot (same for all 5 methods) and the reference trajectory
The root is the starting spot of every method. It comes from, in this order: `ROOT_SPOT` → (`REF_MODE='none'`) `ROOT_REGION` → the region with the largest share of spots of the **first reference state** → a CytoTRACE-like guess (the most "immature-looking" spot = many expressed genes, as in stLearn's `set_root`).
The reference (`REF_MODE='annotation'` or `'markers'`) is used **only to validate** results (section 10), never to build them.

In [ ]:
# [root]  root spot + the optional REFERENCE trajectory (validation only)
def pick_root(pool):
    """stLearn's set_root rule (CytoTRACE-like): the most 'immature-looking' spot (many expressed genes) among the spots in `pool`."""
    sub = ad[pool]
    if sub.n_obs * sub.n_vars > 2e8: sub = sub[:, sub.var['highly_variable']]                 # keep memory in check on big data
    Xs = sub.X.toarray() if hasattr(sub.X, 'toarray') else np.asarray(sub.X)
    n_expr = (Xs > 0).sum(1).astype(float)
    Xc = Xs - Xs.mean(0); nc = n_expr - n_expr.mean()
    corr = (Xc * nc[:, None]).sum(0) / (np.sqrt((Xc ** 2).sum(0) * (nc ** 2).sum()) + 1e-12)
    top = np.argsort(-np.nan_to_num(corr))[:1000]
    return int(np.where(ad.obs_names == sub.obs_names[int(np.argmax(Xs[:, top].mean(1)))])[0][0])

assert REF_MODE in ('none', 'annotation', 'markers'), "REF_MODE must be 'none', 'annotation' or 'markers'"
HAS_REF = REF_MODE in ('annotation', 'markers')
regions_ = ad.obs.groupby('region', observed=True)

if REF_MODE == 'markers':
    assert len(MARKERS) >= 2, 'MARKERS needs at least two states (in trajectory order)'
    REF_STATES = list(MARKERS)
    for name, genes in MARKERS.items():
        g = [x for x in genes if x in ad.var_names]
        if g: sc.tl.score_genes(ad, g, score_name=f'score_{name}', random_state=0)
        else: ad.obs[f'score_{name}'] = 0.0
        print(f'{name:13s} markers found: {g}')
    score_cols = [f'score_{k}' for k in REF_STATES]
    Zs = np.column_stack([((ad.obs[c] - ad.obs[c].mean()) / ad.obs[c].std()).values for c in score_cols])
    clear = Zs.max(1) > 1.0                                   # a spot gets a reference state only if it clearly looks like that area (z > 1)
    first = clear & (Zs.argmax(1) == 0)                       # spots that clearly look like the FIRST state
    frac_first = pd.Series(first, index=ad.obs_names).groupby(ad.obs['region'], observed=True).mean()
    print(f'\nShare of clear "{REF_STATES[0]}" spots per region:', frac_first.round(3).to_dict())
    ROOT_CLUSTER = frac_first.idxmax()
    root_pool = (ad.obs['region'] == ROOT_CLUSTER).values & first
    if root_pool.sum() < 5: root_pool = first                 # safety: too few such spots in that region -> all clear spots of the first state
    ref_state = np.where(clear, np.array(REF_STATES, dtype=object)[Zs.argmax(1)], None)
    ref_label = np.where(clear, np.array(REF_STATES, dtype=object)[Zs.argmax(1)], 'other')
    ad.obs['target_score'] = ad.obs[score_cols[-1]].values.astype(float); TARGET_NAME = f'{REF_STATES[-1]} score'
    cl_scores = regions_[score_cols].mean(); cl_scores['target_score'] = regions_['target_score'].mean()
    print('\nMean marker score per region (score = markers minus matched background genes; 0 = background):'); display(cl_scores.round(3))

elif REF_MODE == 'annotation':
    if ANNOT_FILE: ann = pd.read_csv(ANNOT_FILE, sep=None, engine='python', index_col=0)       # csv or tsv (separator detected)
    else:          ann = ad.obs
    if ANNOT_COLUMN not in ann.columns: raise KeyError(f'ANNOT_COLUMN {ANNOT_COLUMN!r} not found. Available: {list(ann.columns)[:40]}')
    lab = ann[ANNOT_COLUMN].astype(object).reindex(ad.obs_names)
    if lab.notna().sum() == 0:                                # barcodes written differently (with/without the "-1" suffix)?
        strip = lambda ix: pd.Index(ix).astype(str).str.replace(r'-\d+$', '', regex=True)
        m_ = pd.Series(ann[ANNOT_COLUMN].astype(object).values, index=strip(ann.index)); m_ = m_[~m_.index.duplicated()]
        lab = pd.Series(m_.reindex(strip(ad.obs_names)).values, index=ad.obs_names)
    if lab.notna().sum() == 0: raise ValueError(f'No annotation matched the spots. Spot barcodes look like {list(ad.obs_names[:3])}, annotation index like {list(ann.index[:3])}.')
    bad_ = set(STATE_MAP.values()) - set(REF_STATES)
    if bad_: raise ValueError(f'STATE_MAP points to states that are not in REF_STATES: {bad_}')
    ad.obs['annot'] = lab.fillna('unannotated').astype(str).values
    print('Annotation labels in the data:', ad.obs['annot'].value_counts().to_dict())
    miss_ = [k for k in STATE_MAP if k not in set(ad.obs['annot'])]
    if miss_: print('!! STATE_MAP labels that do not occur in the data:', miss_)
    ref_state = np.array([STATE_MAP.get(a_) for a_ in ad.obs['annot']], dtype=object)
    ref_label = ad.obs['annot'].values
    tpos = {s_: i / max(len(REF_STATES) - 1, 1) for i, s_ in enumerate(REF_STATES)}        # target: position in the chain, 0 (first state) ... 1 (last)
    ad.obs['target_score'] = [tpos.get(s_, np.nan) for s_ in ref_state]; TARGET_NAME = ' → '.join(REF_STATES) + ' (annotation)'
    cl_scores = pd.DataFrame({'target_score': regions_['target_score'].mean(),
                              **{f'frac_{s_}': pd.Series(ref_state == s_, index=ad.obs_names).groupby(ad.obs['region'], observed=True).mean() for s_ in REF_STATES}})
    print('\nPer region: mean target and share of every reference state:'); display(cl_scores.round(3))
    ROOT_CLUSTER = cl_scores[f'frac_{REF_STATES[0]}'].idxmax()
    root_pool = (ad.obs['region'] == ROOT_CLUSTER).values & (ref_state == REF_STATES[0])

else:                                                         # no reference
    ref_state = np.full(ad.n_obs, None, dtype=object); ref_label = np.full(ad.n_obs, 'unlabelled', dtype=object)
    REF_STATES = []; ad.obs['target_score'] = np.nan; TARGET_NAME = 'reference (none)'
    cl_scores = pd.DataFrame({'target_score': np.nan}, index=ad.obs['region'].cat.categories)
    ROOT_CLUSTER = None; root_pool = np.ones(ad.n_obs, dtype=bool)
    print('REF_MODE = none -> no reference trajectory; the validation against a reference (section 10) is skipped.')
    print('Spots per expression region:', ad.obs['region'].value_counts().sort_index().to_dict())

# ---- choose the root spot ----
if ROOT_SPOT:
    if ROOT_SPOT not in ad.obs_names: raise KeyError(f'ROOT_SPOT {ROOT_SPOT!r} is not among the kept spots (e.g. {list(ad.obs_names[:3])})')
    root = int(ad.obs_names.get_loc(ROOT_SPOT))
else:
    if REF_MODE == 'none' and ROOT_REGION is not None:
        root_pool = (ad.obs['region'] == str(ROOT_REGION)).values
        if not root_pool.any(): raise ValueError(f'ROOT_REGION {ROOT_REGION!r} does not exist. Regions: {list(ad.obs["region"].cat.categories)}')
    elif REF_MODE == 'none':
        print('!! WARNING: no root given (ROOT_SPOT / ROOT_REGION) and no reference -> the root is the most immature-looking spot of the WHOLE tissue.'
              '\n   Pseudotime needs a biologically meaningful root: set ROOT_SPOT or ROOT_REGION in the settings cell and re-run from this cell.')
    if root_pool.sum() == 0:
        print('!! no spot of the first reference state was found -> root chosen from all spots'); root_pool = np.ones(ad.n_obs, dtype=bool)
    root = pick_root(root_pool)
if ROOT_SPOT or REF_MODE == 'none': ROOT_CLUSTER = str(ad.obs['region'].iloc[root])
ROOT_BARCODE = ad.obs_names[root]
ad.uns['iroot'] = root
ad.obs['ref_state'] = pd.Categorical(ref_state, categories=REF_STATES)
ad.obs['ref_time'] = ad.obs['ref_state'].cat.codes.astype(float).replace(-1.0, np.nan)
ad.obs['ref_label'] = np.asarray(ref_label, dtype=str)
REF_EDGES = list(zip(REF_STATES[:-1], REF_STATES[1:]))
ad.obs['plot_target'] = ad.obs['target_score'].values if HAS_REF else np.log1p(ad.obs['n_genes'].values)       # what the first map panels show
PLOT_NAME = TARGET_NAME if HAS_REF else 'log(genes per spot)'
print('\nreference trajectory:', ' → '.join(REF_STATES) or '(none)', '| spots per reference state:', ad.obs['ref_state'].value_counts().to_dict())
print('root region:', ROOT_CLUSTER, '| root spot:', ROOT_BARCODE)

fig, axes = plt.subplots(1, 3, figsize=(18, 5.5))
s = axes[0].scatter(xy[:, 0], -xy[:, 1], c=ad.obs['plot_target'], s=6, cmap='magma'); plt.colorbar(s, ax=axes[0]); axes[0].set_title(PLOT_NAME)
for k in REF_STATES:
    m = (ad.obs['ref_state'] == k).values; axes[1].scatter(xy[m, 0], -xy[m, 1], s=6, label=k)
if HAS_REF: axes[1].legend(fontsize=8, markerscale=2)
axes[1].set_title('reference states' if HAS_REF else 'no reference states')
for lab_ in ad.obs['region'].cat.categories:
    m = (ad.obs['region'] == lab_).values; axes[2].scatter(xy[m, 0], -xy[m, 1], s=6, label=lab_)
axes[2].legend(fontsize=7, markerscale=2, ncol=2); axes[2].set_title('expression regions (Leiden)')
for a in axes:
    a.scatter(xy[root, 0], -xy[root, 1], c='cyan', s=200, marker='*', edgecolors='k'); a.set_aspect('equal'); a.axis('off')
plt.tight_layout(); plt.show()

def save_pt(col, name):
    pd.DataFrame({'barcode': ad.obs_names, 'x': xy[:, 0], 'y': xy[:, 1], 'region': ad.obs['region'].values,
                  'pseudotime': ad.obs[col].values, 'method': name}).to_csv(f'{OUT}/pt_{name}.csv', index=False)
    print(name, '| root region mean:', round(ad.obs.loc[ad.obs['region'] == ROOT_CLUSTER, col].mean(), 3),
          f'| corr. with {TARGET_NAME}:', round(spearmanr(ad.obs[col], ad.obs['target_score'])[0], 3))

MS_GRAPH = {}       # each method's OWN milestone graph: list of (milestone a, milestone b, weight)
def paga_edges(a, key):
    C_ = a.uns['paga']['connectivities'].toarray(); cats = list(a.obs[key].cat.categories)
    return [(str(cats[i]), str(cats[j]), float(C_[i, j])) for i in range(len(cats)) for j in range(i + 1, len(cats)) if C_[i, j] > 0]

## 1. SpaceFlow (graph neural network + spatial regularisation, then DPT from the root)

In [ ]:
from SpaceFlow import SpaceFlow
import networkx as nx, scipy.sparse as sp
# compatibility fix: SpaceFlow 1.0.4 calls a networkx function that newer networkx removed
if not hasattr(nx, 'to_scipy_sparse_matrix'):
    nx.to_scipy_sparse_matrix = lambda G, format='csr', **kw: sp.csr_matrix(nx.to_scipy_sparse_array(G, format=format, **kw))

EPOCHS = 1000          # SpaceFlow default; early stopping usually ends sooner
sf = SpaceFlow.SpaceFlow(adata=raw.copy())
sf.preprocessing_data(n_top_genes=2000)          # normalise, log, HVGs, PCA, spatial graph (alpha complex)
emb = sf.train(embedding_save_filepath=f'{OUT}/spaceflow_embedding.tsv',
               spatial_regularization_strength=0.1, z_dim=50, lr=1e-3,
               epochs=EPOCHS, max_patience=50, min_stop=100, random_seed=42, gpu=0)
print('embedding shape:', emb.shape)

e = sc.AnnData(emb); e.obs_names = ad.obs_names
sc.pp.neighbors(e, n_neighbors=20, use_rep='X')
sc.tl.diffmap(e, n_comps=15)
e.uns['iroot'] = int(root)
sc.tl.dpt(e)
ad.obs['t_spaceflow'] = rank01(e.obs['dpt_pseudotime'].values)
save_pt('t_spaceflow', 'spaceflow')
# ---- SpaceFlow's own milestones: its spatial domains (SpaceFlow segmentation = Leiden on the embedding) + PAGA between them ----
e2 = e.copy(); sc.pp.neighbors(e2, n_neighbors=50, use_rep='X')
sc.tl.leiden(e2, resolution=1.0, key_added='m', flavor='igraph', n_iterations=2, directed=False, random_state=0)
sc.tl.paga(e2, groups='m')
ad.obs['m_spaceflow'] = e2.obs['m'].astype(str).values; MS_GRAPH['SpaceFlow'] = paga_edges(e2, 'm')
print('SpaceFlow milestones:', ad.obs['m_spaceflow'].nunique())


## 2. SpaTrack (optimal transport: cost = expression distance + physical distance)

In [ ]:
import numpy as np
if not hasattr(np, 'Inf'):
    np.Inf = np.inf          # compatibility fix: spaTrack 0.1.1 uses a name removed in NumPy 2
import spaTrack as spt

ALPHA_SPACE, ALPHA_EXPR = 1, 1
st = ad.copy()
st.obsm['X_spatial'] = xy
st.obs['cluster'] = ad.obs['region'].astype(str)
st.obsp['trans'] = spt.get_ot_matrix(st, data_type='spatial', alpha1=ALPHA_SPACE, alpha2=ALPHA_EXPR)
start_cells = spt.set_start_cells(st, select_way='coordinates', start_point=tuple(xy[root]),
                                  basis='spatial', n_neigh=20)
print('start cells:', len(start_cells), '| their tissues:', st.obs['region'].iloc[list(start_cells)].value_counts().to_dict())
ad.obs['t_spatrack'] = rank01(spt.get_ptime(st, list(start_cells)))
save_pt('t_spatrack', 'spatrack')
# ---- SpaTrack's own milestones: communities of its transport (transition) graph; edges = transported mass between them ----
import scipy.sparse as sp
# The OT transition matrix is DENSE (every spot -> every spot), so Leiden on it is extremely slow on TBI.
# Keep only each spot's K_TR strongest transitions (a sparse 'transport kNN graph'); flows between milestones still use the full matrix.
K_TR = 15
Td = st.obsp['trans']; Td = Td.toarray() if sp.issparse(Td) else np.asarray(Td, dtype=float)
top_ = np.argpartition(-Td, K_TR, axis=1)[:, :K_TR]; rr_ = np.repeat(np.arange(Td.shape[0]), K_TR)
Tk = sp.csr_matrix((Td[rr_, top_.ravel()], (rr_, top_.ravel())), shape=Td.shape)
Asym = ((Tk + Tk.T) / 2).tocsr(); Asym.eliminate_zeros()
sc.tl.leiden(st, adjacency=Asym, resolution=1.0, key_added='m', flavor='igraph', n_iterations=2, directed=False, random_state=0)
labm = st.obs['m'].astype(str).values; ms_ = sorted(set(labm), key=int)
H = sp.csr_matrix((np.ones(len(labm)), (np.arange(len(labm)), [ms_.index(x) for x in labm])), shape=(len(labm), len(ms_)))
F = np.asarray(H.T @ (Td @ H)); F = F + F.T; np.fill_diagonal(F, 0); F = F / (F.max() + 1e-12)
ad.obs['m_spatrack'] = labm
MS_GRAPH['SpaTrack'] = [(ms_[i], ms_[j], float(F[i, j])) for i in range(len(ms_)) for j in range(i + 1, len(ms_)) if F[i, j] > 1e-3]
print('SpaTrack milestones:', len(ms_))


## 3. StaVia (spatial smoothing + spatial kNN, random walks with memory)

In [ ]:
# ---- compatibility fixes (pyVIA was written for NumPy 1 and older SciPy) ----
import numpy as np, scipy.sparse as sp
for old, new in {'bool8':'bool_','object0':'object_','int0':'intp','uint0':'uintp','float_':'float64',
                 'complex_':'complex128','unicode_':'str_','string_':'bytes_','longfloat':'longdouble',
                 'singlecomplex':'complex64','cfloat':'complex128','longcomplex':'clongdouble',
                 'clongfloat':'clongdouble','void0':'void','bytes0':'bytes_','str0':'str_'}.items():
    if old not in np.__dict__: setattr(np, old, getattr(np, new))
for old, val in {'Inf':np.inf,'NaN':np.nan,'Infinity':np.inf,'NINF':-np.inf,'PINF':np.inf}.items():
    if old not in np.__dict__: setattr(np, old, val)
import matplotlib, matplotlib.cm as cm
if not hasattr(cm, 'get_cmap'): cm.get_cmap = matplotlib.colormaps.get_cmap   # removed in new matplotlib
import pyVIA.utils_via as uv, pyVIA.core as vc
def _sparse_from_igraph(graph, weight_attr=None):            # fixed copy of pyVIA's helper (new SciPy rejects zip())
    edges = graph.get_edgelist()
    w = list(graph.es[weight_attr]) if weight_attr else [1] * len(edges)
    if not graph.is_directed():
        edges = edges + [(b, a) for a, b in edges]; w = w + w
    n = graph.vcount()
    if not edges: return sp.csr_matrix((n, n))
    r, c = map(np.asarray, zip(*edges))
    return sp.csr_matrix((np.asarray(w, float), (r, c)), shape=(n, n))
uv.get_sparse_from_igraph = vc.get_sparse_from_igraph = _sparse_from_igraph
from pyVIA.core import VIA
from pyVIA.utils_via import spatial_input
from sklearn.decomposition import PCA

# ---- Method 5: StaVia ----
SPATIAL_WEIGHT = 0.3     # how much each spot's expression is mixed with its 8 spatial neighbours (paper uses 0.3 to 0.5)
Xh = ad[:, ad.var['highly_variable']].X                      # log-normalised HVG expression
Xh = Xh.toarray() if sp.issparse(Xh) else np.asarray(Xh)
Xs = spatial_input(Xh, xy.astype(float), knn_spatial=8, spatial_weight=SPATIAL_WEIGHT)   # step 1: spatial smoothing
Xs = Xs.toarray() if sp.issparse(Xs) else np.asarray(Xs)
pcs_sv = PCA(30, random_state=0).fit_transform((Xs - Xs.mean(0)) / (Xs.std(0) + 1e-8))

v = VIA(data=pcs_sv, true_label=list(ad.obs['region'].astype(str)), root_user=[int(root)],
        knn=30, memory=5, random_seed=42, num_threads=N_THREADS,
        spatial_coords=xy.astype(float), do_spatial_knn=True, spatial_knn=15)                  # step 2: spatial kNN
v.run_VIA()

pt = np.asarray(v.single_cell_pt_markov, dtype=float)
assert len(pt) == ad.n_obs, f'StaVia returned {len(pt)} values for {ad.n_obs} spots'
ad.obs['t_stavia'] = rank01(pt)
ad.obs['stavia_cluster'] = pd.Categorical(np.asarray(v.labels).astype(str))
print('StaVia clusters:', ad.obs['stavia_cluster'].nunique(), '| terminal clusters:', v.terminal_clusters)
save_pt('t_stavia', 'stavia')
ad.obs['m_stavia'] = ad.obs['stavia_cluster'].astype(str).values          # StaVia's own milestones
MS_GRAPH['StaVia'] = [(str(a_), str(b_), 1.0) for a_, b_ in v.edgelist_maxout]  # StaVia's own cluster graph

fig, axes = plt.subplots(1, 2, figsize=(12, 5))
s = axes[0].scatter(xy[:, 0], xy[:, 1], c=ad.obs['t_stavia'], s=5, cmap='viridis'); plt.colorbar(s, ax=axes[0])
axes[0].scatter(xy[root, 0], xy[root, 1], c='red', s=100, marker='*'); axes[0].set_title('StaVia pseudotime')
for k in ad.obs['stavia_cluster'].cat.categories:
    m = (ad.obs['stavia_cluster'] == k).values; axes[1].scatter(xy[m, 0], xy[m, 1], s=5, label=k)
axes[1].legend(fontsize=7, markerscale=3, ncol=2); axes[1].set_title('StaVia clusters')
for a in axes: a.set_aspect('equal')
plt.tight_layout(); plt.show()


## 4. stLearn PSTS (full version with the H&E image; without an image it runs on space + expression only)
Steps as in the stLearn tutorial: normalise/log → PCA → ResNet50 image-tile features → **stSME** → PCA → Louvain → **PSTS** (diffusion pseudotime from our root + cluster-level trajectory). The next cell saves the script, the cell after runs it in the Python 3.11 environment.

In [ ]:
%%writefile run_stlearn_psts.py
import os; os.environ['MPLBACKEND'] = 'Agg'   # this runs outside Jupyter: never inherit the notebook's inline plotting backend
"""stLearn PSTS (Pham et al., Nat Commun 2023) following the official stLearn tutorial.
Runs in its own Python 3.11 environment (stLearn needs Python < 3.13 and NumPy < 2).
Input : a 10x Visium folder (filtered_feature_bc_matrix.h5 + spatial/) or .h5ad, the root spot barcode, and the spots to keep.
Output: CSV with one stLearn pseudotime value per spot (+ its stSME louvain cluster)."""

import argparse, os, json, numpy as np, pandas as pd
ap = argparse.ArgumentParser()
ap.add_argument('--visium_dir', required=True); ap.add_argument('--root_barcode', required=True)
ap.add_argument('--keep', required=True, help='text file with barcodes to keep (same spots as the other methods)')
ap.add_argument('--out', required=True); ap.add_argument('--tile_dir', default='stl_tiles'); ap.add_argument('--count_file', default='filtered_feature_bc_matrix.h5')
ap.add_argument('--resolution', type=float, default=1.0); ap.add_argument('--eps', type=float, default=50)
ap.add_argument('--no_imagenet_weights', action='store_true', help='testing only: offline, use untrained CNN')
a = ap.parse_args()

if a.no_imagenet_weights:                      # offline test only; on Kaggle the ImageNet ResNet50 weights download normally
    import keras.applications.resnet50 as kr
    _R = kr.ResNet50; kr.ResNet50 = lambda *x, **k: _R(*x, **{**k, 'weights': None})
import stlearn as st, scanpy as sc

if a.visium_dir.endswith('.h5ad'):                  # data that is not a Visium folder: the h5ad written by the notebook (raw counts + coordinates [+ image])
    data = sc.read_h5ad(a.visium_dir)
    xy_ = np.asarray(data.obsm['spatial'], dtype=float)
    scalef = 1.0
    for v_ in (data.uns.get('spatial') or {}).values():
        imgs_ = v_.get('images', {})
        q_ = v_.setdefault('use_quality', 'hires' if 'hires' in imgs_ else 'lowres')
        if q_ in imgs_:
            arr_ = np.asarray(imgs_[q_])
            if np.issubdtype(arr_.dtype, np.floating) and arr_.max() <= 1.0:
                imgs_[q_] = np.clip(arr_ * 255.0, 0, 255).astype(np.uint8)
        scalef = float((v_.get('scalefactors') or {}).get(f'tissue_{q_}_scalef', 1.0))
        break
    data.obs['imagecol'], data.obs['imagerow'] = xy_[:, 0] * scalef, xy_[:, 1] * scalef
else:
    data = st.Read10X(a.visium_dir, count_file=a.count_file)
data.var_names_make_unique()
HAS_IMG = any(len(v_.get('images', {})) > 0 for v_ in (data.uns.get('spatial') or {}).values())
keep = [l.strip() for l in open(a.keep) if l.strip()]
data = data[data.obs_names.isin(keep)].copy()
data.layers['raw_count'] = data.X.copy()

# 1) preprocessing (stLearn tutorial)
st.pp.filter_genes(data, min_cells=3)
st.pp.normalize_total(data); st.pp.log1p(data)
st.em.run_pca(data, n_comps=50, random_state=0)

# 2) morphology: cut an image tile around every spot and describe it with a ResNet50 CNN
if HAS_IMG:
    os.makedirs(a.tile_dir, exist_ok=True)
    st.pp.tiling(data, a.tile_dir)
    st.pp.extract_feature(data)
    SME_WEIGHTS = 'weights_matrix_all'
else:
    print('no H&E image in the data -> stSME uses physical distance only (no morphology)')
    SME_WEIGHTS = 'physical_distance'

# 3) stSME: smooth each spot with neighbours that are close in space AND look alike in the image AND in expression
st.spatial.SME.SME_normalize(data, use_data='raw', weights=SME_WEIGHTS)
d = data.copy(); d.X = d.obsm['raw_SME_normalized']
st.pp.scale(d); st.em.run_pca(d, n_comps=50, random_state=0)
st.pp.neighbors(d, n_neighbors=25, use_rep='X_pca', random_state=0)
st.tl.clustering.louvain(d, resolution=a.resolution, random_state=0)

# 4) PSTS: root = the same root spot used by all methods; diffusion pseudotime on the stSME graph + spatial sub-clusters
root_idx = int(np.where(d.obs_names == a.root_barcode)[0][0])
d.uns['iroot'] = root_idx
st.spatial.trajectory.pseudotime(d, eps=a.eps, use_rep='X_pca', use_label='louvain', max_nodes=5)   # allow 4-cluster paths like the paper's
root_cl = str(d.obs['louvain'].iloc[root_idx])
info = {'image_used': bool(HAS_IMG), 'n_spots': int(d.n_obs), 'n_louvain': int(d.obs['louvain'].nunique()), 'root_cluster': root_cl}

# 5) PSTS global trajectory (cluster level), as in the paper: a path of connected clusters starting at the root's cluster.
#    stLearn lists the valid paths in uns['available_paths']; we try the longest ones that start at the root cluster.
paths = [p for p in d.uns.get('available_paths', {}).values() if str(p[0]) == root_cl and len(p) > 1]
cl_pt = d.obs.groupby('louvain', observed=True)['dpt_pseudotime'].mean()
paths.sort(key=lambda p: (len(p), cl_pt.get(str(p[-1]), 0)), reverse=True)
for p in paths:
    try:
        st.spatial.trajectory.pseudotimespace_global(d, use_label='louvain', list_clusters=[str(c) for c in p])
        info['psts_path'] = [str(c) for c in p]; info.pop('psts_global_error', None); break
    except Exception as e:
        info['psts_global_error'] = repr(e)[:200]
else:
    info.setdefault('psts_global_error', 'no available path from the root cluster')

import scanpy as sc
if 'paga' not in d.uns or d.uns['paga'].get('groups') != 'louvain': sc.tl.paga(d, groups='louvain')
Cg = d.uns['paga']['connectivities'].toarray(); cats = [str(c) for c in d.obs['louvain'].cat.categories]
info['paga_edges'] = [[cats[i], cats[j], float(Cg[i, j])] for i in range(len(cats)) for j in range(i + 1, len(cats)) if Cg[i, j] > 0]
pd.DataFrame({'barcode': d.obs_names, 'stlearn_pseudotime': d.obs['dpt_pseudotime'].values,
              'stlearn_cluster': d.obs['louvain'].astype(str).values}).to_csv(a.out, index=False)
json.dump(info, open(a.out.replace('.csv', '_info.json'), 'w'), indent=1)
print('stLearn PSTS done:', {k: v for k, v in info.items() if k != 'paga_edges'})

In [ ]:
# [stlearn-run]  run stLearn PSTS in its own environment with live progress (skipped if a result for these spots already exists)
import json
if not os.path.exists(f'{OUT}/raw_stlearn.csv'):
    if INPUT_KIND == 'visium': STL_INPUT = VISIUM_DIR
    else:                                                                    # .h5ad input: give stLearn raw counts + coordinates (+ the H&E image if the data has one)
        STL_INPUT = f'{OUT}/stlearn_input.h5ad'; m_ = mini_adata(raw)
        if 'spatial' in raw.uns: m_.uns['spatial'] = raw.uns['spatial']
        m_.write(STL_INPUT)
    cmd = [STL_PY, 'run_stlearn_psts.py', '--visium_dir', STL_INPUT, '--count_file', VISIUM_COUNT_FILE, '--root_barcode', ROOT_BARCODE,
           '--keep', f'{OUT}/kept_spots.txt', '--out', f'{OUT}/raw_stlearn.csv', '--tile_dir', STL_TILES]
    rc, tail = run_live(cmd, skip=lambda l: ('it/s' in l or 's/it' in l) and '100%' not in l, timeout_min=STL_MAX_MIN)   # hides progress-bar spam
    if rc != 0: print(tail); print('stLearn did not finish -> skipped (the comparison will run without it)')
if os.path.exists(f'{OUT}/raw_stlearn.csv'):
    res = pd.read_csv(f'{OUT}/raw_stlearn.csv', index_col='barcode')
    print('spots returned:', len(res), 'of', ad.n_obs)
    ad.obs['t_stlearn'] = rank01(res['stlearn_pseudotime'].reindex(ad.obs_names).values)
    ad.obs['stlearn_cluster'] = res['stlearn_cluster'].reindex(ad.obs_names).astype(str).values
    info = json.load(open(f'{OUT}/raw_stlearn_info.json')); print({k: v for k, v in info.items() if k != 'paga_edges'})
    ad.obs['m_stlearn'] = ad.obs['stlearn_cluster']                          # stLearn's own milestones (stSME clusters)
    MS_GRAPH['stLearn'] = [tuple(e_) for e_ in info.get('paga_edges', [])]   # stLearn's PAGA graph between them
    save_pt('t_stlearn', 'stlearn')


## 5. CASCAT (causal trajectories; Yu et al., NAR 2025)
Run as in the authors' README (`main.py --mode train`) with their Visium settings (`CASCAT_YAML`). CASCAT needs region labels (our Leiden regions), the root cluster and the root spot. Output: `cascat_pseudotime` per spot.

**Runtime note:** almost all of CASCAT's time is the causal-tree step. By default the binned CMI estimator is used (`CASCAT_KDE = 0`, minutes; the authors' kernel-density estimator takes hours), 2 trials × 50 epochs, and a time limit after which CASCAT is skipped. For the authors' exact settings set `CASCAT_KDE = 1`, `CASCAT_TRIALS, CASCAT_EPOCHS = 5, 100`.

In [ ]:
%%writefile run_cascat.py
import os; os.environ['MPLBACKEND'] = 'Agg'   # this runs outside Jupyter: never inherit the notebook's inline plotting backend
"""CASCAT (Yu et al., Nucleic Acids Research 2025) on one spatial section, following the authors' README.
Runs inside the CASCAT folder, in its own Python 3.10 environment.
Input : an h5ad with raw counts in X, obsm['spatial'], obs['cluster'] (region labels) and uns['start_id'] (root spot).
Output: CSV with CASCAT's per-spot pseudotime (adata.obs['cascat_pseudotime']) and its clusters."""
import argparse, os, sys, subprocess, yaml, glob
ap = argparse.ArgumentParser()
ap.add_argument('--cascat_dir', required=True); ap.add_argument('--h5ad', required=True)
ap.add_argument('--root_cluster', required=True); ap.add_argument('--out', required=True)
ap.add_argument('--work', default='cascat_run'); ap.add_argument('--device', default='cuda')
ap.add_argument('--ntrials', type=int, default=5); ap.add_argument('--epochs', type=int, default=100)
ap.add_argument('--kde', type=int, default=1, help='1 = authors default (kernel-density CMI in the tree step, slow); 0 = CASCAT binned CMI estimator (fast)')
ap.add_argument('--species', default='Human'); ap.add_argument('--yaml', default='HER2ST.yml'); ap.add_argument('--data_name', default='data')
a = ap.parse_args()

cfg = yaml.safe_load(open(os.path.join(a.cascat_dir, 'config/yaml/CMI', a.yaml)))   # authors' Visium settings
cfg.update({'data_name': a.data_name, 'adata_file': os.path.abspath(a.h5ad), 'root': str(a.root_cluster),
            'device': a.device, 'ntrials': a.ntrials, 'epochs': a.epochs, 'species': a.species,
            'img_dir': f'{a.work}/img/', 'clu_dir': f'{a.work}/clu/', 'output_dir': f'{a.work}/result/',
            'CMI_dir': f'{a.work}/cmi/'})
cfg.setdefault('ncomp', 15)                      # diffusion components for pseudotime (value from the authors' tree1.yml)
os.makedirs(a.work, exist_ok=True); os.makedirs(cfg['img_dir'], exist_ok=True)
yml = f'{a.work}/{a.data_name}.yml'; yaml.safe_dump(cfg, open(yml, 'w'))
print('CASCAT config:', {k: cfg[k] for k in ['a_k', 'percent', 'epochs', 'ntrials', 'root', 'hvg', 'norm_target', 'device']})

# CASCAT hard-codes kde=True in infer.py; make it follow --kde (authors' own alternative estimator in BinaryCMITree)
inf = os.path.join(a.cascat_dir, 'infer.py'); src = open(inf).read()
if 'CASCAT_KDE' not in src:
    src = 'import os\n' + src.replace("save_dir=self.save_path, kde=True)", "save_dir=self.save_path, kde=os.environ.get('CASCAT_KDE', '1') == '1')")
    open(inf, 'w').write(src)
os.environ['CASCAT_KDE'] = str(a.kde)
r = subprocess.run([sys.executable, 'main.py', '--YAML', yml, '--mode', 'train', '--verbose', 'True'], cwd=a.cascat_dir, env={**os.environ, 'MPLBACKEND': 'Agg'})
if r.returncode != 0: sys.exit(r.returncode)

import scanpy as sc, pandas as pd
res = sc.read_h5ad(os.path.join(cfg['output_dir'], 'data_processed.h5ad'))
pd.DataFrame({'barcode': res.obs_names, 'cascat_pseudotime': res.obs['cascat_pseudotime'].values,
              'cascat_cluster': res.obs['cascat_clusters'].astype(str).values}).to_csv(a.out, index=False)
print('CASCAT done:', res.n_obs, 'spots | root cluster:', res.uns.get('root'),
      '| tree:', res.uns['cascat_connectivities'].shape if 'cascat_connectivities' in res.uns else None)


In [ ]:
# [cascat-run]  run CASCAT in its own environment (settings CASCAT_* are in the settings cell)
ep = [0]
def quiet(l):
    if l.startswith('Epoch'): ep[0] += 1; return ep[0] % 10 != 1
    if l.startswith('leaf_i'): ep[0] += 1; return ep[0] % 20 != 1          # show every 20th tree calculation
    return ('Warning' in l) or ('warnings.warn' in l) or ('it/s' in l and '100%' not in l)

if not os.path.exists(f'{OUT}/raw_cascat.csv'):
    cin = mini_adata(raw)
    cin.obs['cluster'] = ad.obs['region'].astype(str).values
    cin.obsm['spatial'] = xy.copy(); cin.uns = {'start_id': ROOT_BARCODE}
    cin.write(f'{OUT}/cascat_input.h5ad')
    cmd = [CAS_PY, 'run_cascat.py', '--cascat_dir', CASCAT_DIR, '--h5ad', f'{OUT}/cascat_input.h5ad',
           '--root_cluster', str(ROOT_CLUSTER), '--out', f'{OUT}/raw_cascat.csv', '--work', CASCAT_WORK,
           '--device', CASCAT_DEVICE, '--ntrials', str(CASCAT_TRIALS), '--epochs', str(CASCAT_EPOCHS), '--kde', str(CASCAT_KDE),
           '--species', SPECIES, '--yaml', CASCAT_YAML, '--data_name', re.sub(r'\W+', '_', RUN_NAME)]
    shutil.rmtree(f'{CASCAT_WORK}/result', ignore_errors=True)               # clear the previous run's result
    rc, tail = run_live(cmd, skip=quiet, timeout_min=CASCAT_MAX_MIN, done_file=CASCAT_RESULT)
    if rc != 0: print(tail[-2000:]); print('CASCAT did not finish -> skipped (the comparison will run without it)')
    if os.path.exists(CASCAT_RESULT):
        shutil.copy(CASCAT_RESULT, f'{OUT}/cascat_result.h5ad')   # keep CASCAT's tree
h5_res = CASCAT_RESULT
if not os.path.exists(f'{OUT}/raw_cascat.csv') and os.path.exists(h5_res):     # rescue: build the CSV from CASCAT's saved result
    r_ = sc.read_h5ad(h5_res)
    pd.DataFrame({'barcode': r_.obs_names, 'cascat_pseudotime': r_.obs['cascat_pseudotime'].values,
                  'cascat_cluster': r_.obs['cascat_clusters'].astype(str).values}).to_csv(f'{OUT}/raw_cascat.csv', index=False)
if os.path.exists(f'{OUT}/raw_cascat.csv'):
    res = pd.read_csv(f'{OUT}/raw_cascat.csv', index_col='barcode')
    ad.obs['t_cascat'] = rank01(res['cascat_pseudotime'].reindex(ad.obs_names).values)
    ad.obs['cascat_cluster'] = res['cascat_cluster'].reindex(ad.obs_names).astype(str).values
    save_pt('t_cascat', 'cascat')
    ad.obs['m_cascat'] = ad.obs['cascat_cluster']                            # CASCAT's own milestones
    h5c = f'{OUT}/cascat_result.h5ad' if os.path.exists(f'{OUT}/cascat_result.h5ad') else CASCAT_RESULT
    if os.path.exists(h5c):
        Tc = sc.read_h5ad(h5c).uns['cascat_connectivities']; Tc = Tc if isinstance(Tc, pd.DataFrame) else pd.DataFrame(Tc)
        MS_GRAPH['CASCAT'] = [(str(r_), str(c_), 1.0) for r_ in Tc.index for c_ in Tc.columns
                              if str(r_) < str(c_) and (Tc.loc[r_, c_] > 0 or Tc.loc[c_, r_] > 0)]   # CASCAT's causal tree



## 6. Comparison
1. **Biological check** — does each method start in the root region and increase towards the end region (needs a reference)?
2. **Mean pseudotime per region**.
3. **Agreement matrix** (Spearman) and **M_eff** (how many truly different opinions).
4. **Maps** of every method (cyan star = root).

In [ ]:
METHODS = [  # column, name, family
    ('t_spaceflow', 'SpaceFlow', 'Graph neural net (DGI) + DPT'),
    ('t_spatrack',  'SpaTrack',  'Optimal transport (expression + distance)'),
    ('t_stavia',    'StaVia',    'Random walks with memory + spatial smoothing'),
    ('t_stlearn',   'stLearn',   'stSME (space + image + expression) + PSTS'),
    ('t_cascat',    'CASCAT',    'Causal (CMI) pruned spatial graph + tree'),
]
METHODS = [m for m in METHODS if m[0] in ad.obs]
cols = [m[0] for m in METHODS]; names = [m[1] for m in METHODS]
print(f'{len(cols)} methods ran:', ', '.join(names))

END_REGION = cl_scores['target_score'].idxmax() if HAS_REF else None
# 1) biological check: the trajectory should START in the root region and END in the region with the highest target value
chk = pd.DataFrame({
    'root cluster mean pt':   [ad.obs.loc[ad.obs['region'] == ROOT_CLUSTER, c].mean() for c in cols],
    'end region mean pt': [ad.obs.loc[ad.obs['region'] == END_REGION, c].mean() if END_REGION is not None else np.nan for c in cols],
    f'Spearman with {TARGET_NAME}': [spearmanr(ad.obs[c], ad.obs['target_score'])[0] for c in cols],
    'family': [m[2] for m in METHODS]}, index=names).round(3)
print(f'\n1) Does each method run from the root region {ROOT_CLUSTER} (should be ≈0) towards the region with the highest {TARGET_NAME} (should be ≈1)?'); display(chk)

# 2) mean pseudotime of every region cluster, clusters sorted from least to most injured
order = cl_scores['target_score'].sort_values().index
reg = ad.obs.groupby('region', observed=True)[cols].mean().loc[order]; reg.columns = names
reg.insert(0, TARGET_NAME, cl_scores.loc[order, 'target_score'].values)
print(f'\n2) Mean pseudotime per region cluster (rows: lowest -> highest {TARGET_NAME}):'); display(reg.round(3))

# 3) agreement + M_eff
A = ad.obs[cols].corr(method='spearman'); A.index = A.columns = names
lam = np.clip(np.linalg.eigvalsh(A.values), 0, None); meff = lam.sum() ** 2 / (lam ** 2).sum()
fig, ax = plt.subplots(figsize=(1.1 * len(names) + 2, 1.1 * len(names) + 1))
im = ax.imshow(A.values, cmap='viridis', vmin=min(0, A.values.min()), vmax=1)
ax.set_xticks(range(len(names))); ax.set_xticklabels(names, rotation=45, ha='right')
ax.set_yticks(range(len(names))); ax.set_yticklabels(names)
for i in range(len(names)):
    for j in range(len(names)):
        ax.text(j, i, f'{A.values[i, j]:.2f}', ha='center', va='center', color='black' if A.values[i, j] > 0.6 else 'white')
plt.colorbar(im, ax=ax, fraction=0.046, label='Spearman agreement'); ax.set_title(f'3) Agreement   (M_eff = {meff:.2f} of {len(names)})')
plt.tight_layout(); plt.show()
red = ((A.sum(1) - 1) / (len(names) - 1)).sort_values(ascending=False).round(3)
print('M_eff =', round(meff, 2), '\nAverage agreement with the other methods:'); print(red.to_string())

# 4) maps
n = len(cols) + 2; nc = 4; nr = int(np.ceil(n / nc))
fig, axes = plt.subplots(nr, nc, figsize=(5 * nc, 4.6 * nr)); axes = np.ravel(axes)
for lab in ad.obs['region'].cat.categories:
    m = (ad.obs['region'] == lab).values; axes[0].scatter(xy[m, 0], -xy[m, 1], s=4, label=lab)
axes[0].legend(fontsize=6, markerscale=3, ncol=2); axes[0].set_title('Region clusters')
s0 = axes[1].scatter(xy[:, 0], -xy[:, 1], c=ad.obs['plot_target'], s=4, cmap='magma'); axes[1].set_title(PLOT_NAME)
for a, c, nm in zip(axes[2:], cols, names):
    s = a.scatter(xy[:, 0], -xy[:, 1], c=ad.obs[c], s=4, cmap='viridis', vmin=0, vmax=1); a.set_title(nm)
for a in axes[:n]:
    a.scatter(xy[root, 0], -xy[root, 1], c='cyan', s=150, marker='*', edgecolors='k'); a.set_aspect('equal'); a.axis('off')
for a in axes[n:]: a.axis('off')
fig.colorbar(s, ax=axes[:n].tolist(), fraction=0.02, label='pseudotime (0 = root, 1 = latest)'); plt.show()

chk.to_csv(f'{OUT}/comparison_check.csv'); reg.to_csv(f'{OUT}/comparison_regions.csv'); A.round(4).to_csv(f'{OUT}/comparison_agreement.csv')
ad.obs[['region', 'target_score'] + cols].assign(x=xy[:, 0], y=xy[:, 1]).to_csv(f'{OUT}/all_methods_pseudotime.csv')
print('Saved:', sorted(os.path.basename(f) for f in glob.glob(f'{OUT}/*.csv')))


## 6b. Pseudotime and trajectory graph for each method
Three panels per method (also saved as `fig_<method>.png`): **(a)** pseudotime on the H&E image (blank canvas if the data has none), **(b)** trajectory graph on the tissue (StaVia: its cluster graph; stLearn: its PSTS path; CASCAT: its causal tree; SpaceFlow and SpaTrack: the shared region graph directed by their pseudotime), **(c)** pseudotime per region.

In [ ]:
# ---------- Per-method figures: (a) pseudotime on the H&E image, (b) trajectory graph on the tissue, (c) pseudotime per region ----------
import networkx as nx, scanpy as sc
from matplotlib.patches import FancyArrowPatch

img, pxy = IMG, PXY                                              # H&E image (or a blank canvas if the data has none) and the spot positions on it

if 'paga' not in ad.uns or ad.uns['paga'].get('groups') != 'region':
    sc.tl.paga(ad, groups='region')                               # shared expression graph between regions (used if a method has no graph of its own)
PAGA_EDGES = [(ad.obs['region'].cat.categories[i], ad.obs['region'].cat.categories[j])
              for i, j in zip(*np.triu(ad.uns['paga']['connectivities'].toarray() > 0.1, 1).nonzero())]

def medoid_positions(labels):
    labels = pd.Series(np.asarray(labels).astype(str), index=ad.obs_names); pos = {}
    for k, idx in labels.groupby(labels).groups.items():
        p = pxy[ad.obs_names.get_indexer(idx)]; pos[k] = p[np.argmin(((p - p.mean(0)) ** 2).sum(1))]
    return labels, pos

def plot_method(col, name, labels=None, edges=None, graph_title=None):
    if col not in ad.obs: print(name, 'not run -> no plot'); return
    t = ad.obs[col].values
    labels, pos = medoid_positions(ad.obs['region'] if labels is None else labels)
    edges = PAGA_EDGES if edges is None else edges
    cl_t = pd.Series(t, index=labels.index).groupby(labels).mean()
    fig, ax = plt.subplots(1, 3, figsize=(21, 6.5), gridspec_kw={'width_ratios': [1, 1, 1.1]})
    # (a) pseudotime on H&E
    ax[0].imshow(img); s = ax[0].scatter(pxy[:, 0], pxy[:, 1], c=t, cmap='viridis', s=9, vmin=0, vmax=1, alpha=.9)
    ax[0].scatter(*pxy[root], c='cyan', s=220, marker='*', edgecolors='k'); plt.colorbar(s, ax=ax[0], fraction=.04, label='pseudotime')
    ax[0].set_title(f'{name}: pseudotime (0 = root, 1 = latest)')
    # (b) trajectory graph: nodes = clusters at their tissue position, arrows from earlier to later cluster
    ax[1].imshow(img, alpha=.45); ax[1].scatter(pxy[:, 0], pxy[:, 1], c=t, cmap='viridis', s=4, vmin=0, vmax=1, alpha=.25)
    for a, b in edges:
        a, b = str(a), str(b)
        if a not in pos or b not in pos: continue
        if cl_t[a] > cl_t[b]: a, b = b, a
        ax[1].add_patch(FancyArrowPatch(pos[a], pos[b], arrowstyle='-|>', mutation_scale=18, lw=2.2, color='k', shrinkA=12, shrinkB=12))
    sizes = labels.value_counts()
    ax[1].scatter([pos[k][0] for k in cl_t.index], [pos[k][1] for k in cl_t.index], c=cl_t.values, cmap='viridis', vmin=0, vmax=1,
                  s=[60 + 3 * sizes[k] ** .8 for k in cl_t.index], edgecolors='k', zorder=3)
    for k in cl_t.index: ax[1].text(*pos[k], k, ha='center', va='center', fontsize=8, color='w', weight='bold', zorder=4)
    ax[1].set_title(graph_title or f'{name}: trajectory graph (region clusters, arrows early → late)')
    # (c) pseudotime per region, regions sorted by their mean
    reg_order = ad.obs.groupby('region', observed=True)[col].mean().sort_values().index
    data = [ad.obs.loc[ad.obs['region'] == r, col].values for r in reg_order]
    ax[2].boxplot(data, vert=False, showfliers=False); ax[2].set_yticks(range(1, len(reg_order) + 1))
    ax[2].set_yticklabels([f"{r}{'  (root)' if r == ROOT_CLUSTER else ''}" for r in reg_order])
    ax[2].set_xlabel('pseudotime'); ax[2].set_title('pseudotime per region (sorted)')
    for a in ax[:2]: a.axis('off')
    plt.tight_layout(); plt.savefig(f'{OUT}/fig_{name}.png', dpi=110, bbox_inches='tight'); plt.show()

# SpaceFlow & SpaTrack build no cluster graph of their own -> shared expression (PAGA) graph between regions, directed by their pseudotime
plot_method('t_spaceflow', 'SpaceFlow', graph_title='SpaceFlow: region graph (PAGA) directed by SpaceFlow pseudotime')
plot_method('t_spatrack',  'SpaTrack',  graph_title='SpaTrack: region graph (PAGA) directed by SpaTrack pseudotime')

# StaVia: its own cluster graph (edgelist_maxout over its own clusters)
try:
    plot_method('t_stavia', 'StaVia', labels=np.asarray(v.labels), edges=[tuple(e) for e in v.edgelist_maxout],
                graph_title="StaVia: its own cluster graph (clusters from StaVia)")
except Exception as e:
    print('StaVia own graph unavailable (', repr(e)[:80], ') -> shared graph'); plot_method('t_stavia', 'StaVia')

# stLearn: its PSTS cluster path (from the stLearn run) over its own Louvain clusters
try:
    info = json.load(open(f'{OUT}/raw_stlearn_info.json')); path = info.get('psts_path', [])
    plot_method('t_stlearn', 'stLearn', labels=ad.obs['stlearn_cluster'].values, edges=list(zip(path[:-1], path[1:])) or None,
                graph_title=f"stLearn: PSTS path {' → '.join(path)}" if path else None)
except Exception as e:
    print('stLearn path unavailable (', repr(e)[:80], ')'); plot_method('t_stlearn', 'stLearn')

# CASCAT: its causal cluster tree
try:
    cres = sc.read_h5ad(CASCAT_RESULT)
    T = cres.uns['cascat_connectivities']; T = pd.DataFrame(T) if not isinstance(T, pd.DataFrame) else T
    tree_edges = [(str(T.index[i]), str(T.columns[j])) for i, j in zip(*np.nonzero(T.values))]
    plot_method('t_cascat', 'CASCAT', labels=ad.obs['cascat_cluster'].values, edges=tree_edges,
                graph_title='CASCAT: its causal cluster tree (clusters from CASCAT)')
except Exception as e:
    print('CASCAT tree unavailable (', repr(e)[:80], ')'); plot_method('t_cascat', 'CASCAT')


## 8. SPACECONS — one consensus pseudotime and one trajectory graph from all five methods
1. **Canonical form + orientation** — rank-scaled pseudotime 0–1, root = 0.
2. **Weights** — leave-one-out agreement (the brief's `S·C·(1−SRI)` is kept in the table for comparison only).
3. **SPACECONS-R** — weighted Borda per spot = the consensus pseudotime; checked against uniform Borda, Kemeny, MC4 and Robust Rank Aggregation.
4. **Uncertainty** — bootstrap over methods (500×): per-spot 95% interval and local ordering confidence.
5. **SPACECONS-G** — majority-rule consensus tree on the shared regions; Topology Disagreement Index (> 0.5 → topology flagged as unresolved).
6. **Evaluation** — consensus vs each method and baselines.

Outputs: `spacecons_pseudotime.csv`, `spacecons_weights.csv`, `spacecons_graph_edges.csv`, `spacecons_evaluation.csv`, `spacecons_summary.json`, `fig_SPACECONS.png`.

In [ ]:
# =====================  SPACECONS consensus: one pseudotime + one trajectory graph from all methods  =====================
import json, itertools, networkx as nx, scanpy as sc
from scipy.stats import rankdata, beta
from scipy.interpolate import RBFInterpolator
from scipy.optimize import milp, LinearConstraint, Bounds
from sklearn.neighbors import NearestNeighbors
from sklearn.model_selection import KFold
rng = np.random.default_rng(0)

METH = [(c, n) for c, n in [('t_spaceflow', 'SpaceFlow'), ('t_spatrack', 'SpaTrack'), ('t_stavia', 'StaVia'),
                            ('t_stlearn', 'stLearn'), ('t_cascat', 'CASCAT')] if c in ad.obs]
cols = [c for c, _ in METH]; names = [n for _, n in METH]
T = np.column_stack([rank01(ad.obs[c].values) for c in cols])          # canonical form: rank-scaled 0..1, oriented root = 0
n, M = T.shape
print(f'{M} methods combined: {", ".join(names)}')

# ---------- Step 2: orientation check (every method must start at the root) ----------
for j, nm in enumerate(names):
    if T[(ad.obs['region'] == ROOT_CLUSTER).values, j].mean() > 0.5: T[:, j] = 1 - T[:, j]; print(f'  {nm} flipped so that the root is early')   # judged on the whole root region, not one spot

# ---------- Weights  w_m ∝ S_m · C_m · (1 − SRI_m)  (brief, step 3) ----------
# S: self-reproducibility. True bootstrap reruns of every method are expensive (step 6); here a cheap proxy:
#    how well each spot's pseudotime is reproduced by its expression neighbours (leave-self-out kNN average on PCA).
# C: spatial coherence = Moran's I of the pseudotime on the spatial kNN graph.
# SRI: R² of a smooth spatial field f(z) (thin-plate spline, smoothing chosen by 5-fold CV) fitted to the pseudotime.
Xp = ad.obsm['X_pca'][:, :30]
ie = NearestNeighbors(n_neighbors=16).fit(Xp).kneighbors(Xp, return_distance=False)[:, 1:]
isp = NearestNeighbors(n_neighbors=7).fit(xy).kneighbors(xy, return_distance=False)[:, 1:]
def morans_I(t, nb):
    z = t - t.mean(); return (z[:, None] * z[nb]).sum() / nb.size / (z ** 2).mean()
Z = (xy - xy.mean(0)) / xy.std(0)
def sri(t, smooth_grid=(1e-3, 1e-2, 1e-1, 1, 10)):
    best, best_s = -np.inf, None
    for s in smooth_grid:                                                   # choose smoothness by cross-validation (GCV stand-in)
        r2 = []
        for tr, te in KFold(5, shuffle=True, random_state=0).split(Z):
            f = RBFInterpolator(Z[tr], t[tr], kernel='thin_plate_spline', smoothing=s, neighbors=200)
            r2.append(1 - ((t[te] - f(Z[te])) ** 2).sum() / ((t[te] - t[te].mean()) ** 2).sum())
        if np.mean(r2) > best: best, best_s = np.mean(r2), s
    return float(np.clip(best, 0, 1)), best_s          # out-of-fold R² (in-sample R² is ~1 for any field once the spline nearly interpolates)
W = []
for j, nm in enumerate(names):
    t = T[:, j]
    S = max(spearmanr(t, t[ie].mean(1))[0], 0)
    C = max(morans_I(t, isp), 0)
    R, s = sri(t)
    W.append((nm, S, C, R, s))
Wt = pd.DataFrame(W, columns=['method', 'S (self-consistency)', 'C (Moran I)', 'SRI', 'TPS smoothing']).set_index('method')
Wt['raw weight S·C·(1−SRI)'] = Wt['S (self-consistency)'] * Wt['C (Moran I)'] * (1 - Wt['SRI'])
Wt['brief weight'] = Wt['raw weight S·C·(1−SRI)'] / Wt['raw weight S·C·(1−SRI)'].sum()
# NEW weights: leave-one-out agreement. Each method is compared with the average of the OTHER methods (it never sees itself);
# a method that disagrees with the rest is down-weighted. Squared so that an outlier method is clearly down-weighted.
# (The brief's S·C·(1−SRI) weight is kept in the table for comparison, but is no longer used: it rewards non-smooth maps,
#  which is not the same as being correct, and lost to equal weights on both datasets.)
Wt['LOO agreement'] = [max(spearmanr(T[:, j], np.delete(T, j, axis=1).mean(1))[0], 0) for j in range(M)]
Wt['weight'] = Wt['LOO agreement'] ** 2 / (Wt['LOO agreement'] ** 2).sum()
w = Wt['weight'].values
print('\nMethod weights (higher = more trusted):'); display(Wt.round(3))

# ---------- SPACECONS-R: consensus ordering ----------
borda_w = rank01((T * w).sum(1))                       # weighted Borda = weighted mean rank (primary estimator)
borda_u = rank01(T.mean(1))                            # uniform Borda (baseline the weighting must beat)
# Robust Rank Aggregation (Kolde 2012): is a spot consistently EARLY (or LATE) across methods?
def rra_p(R):
    Rs = np.sort(R, 1); k = np.arange(1, R.shape[1] + 1)
    return np.minimum(beta.cdf(Rs, k, R.shape[1] - k + 1).min(1) * R.shape[1], 1)
p_early, p_late = rra_p(T), rra_p(1 - T)
# milestone level (regions) for Kemeny and MC4
reg = ad.obs['region'].astype(str).values; K_lab = sorted(set(reg), key=lambda s: int(s) if s.isdigit() else s)
K = len(K_lab); Tm = np.array([[T[reg == k, j].mean() for j in range(M)] for k in K_lab])    # K x M milestone pseudotimes
P = np.zeros((K, K))                                   # P[a,b] = weighted share of methods putting milestone a before b
for j in range(M): P += w[j] * (Tm[:, j][:, None] < Tm[:, j][None, :])
# Kemeny (exact, integer programming over pairwise orders with transitivity constraints)
idx = {(a, b): i for i, (a, b) in enumerate((a, b) for a in range(K) for b in range(K) if a != b)}
c = np.zeros(len(idx)); A, lb, ub = [], [], []
for (a, b), i in idx.items(): c[i] = -P[a, b]
for a in range(K):
    for b in range(a + 1, K):
        row = np.zeros(len(idx)); row[idx[a, b]] = row[idx[b, a]] = 1; A.append(row); lb.append(1); ub.append(1)
for a, b, d in itertools.permutations(range(K), 3):
    row = np.zeros(len(idx)); row[idx[a, b]] = row[idx[b, d]] = row[idx[d, a]] = 1; A.append(row); lb.append(0); ub.append(2)
sol = milp(c, constraints=LinearConstraint(np.array(A), lb, ub), integrality=np.ones(len(idx)), bounds=Bounds(0, 1))
wins = np.zeros(K)
for (a, b), i in idx.items(): wins[a] += round(sol.x[i]) if sol.success else P[a, b]
kemeny_order = [K_lab[i] for i in np.argsort(-wins)]                                     # earliest first
# MC4 (Dwork et al. 2001): random walk that moves from a milestone to one ranked earlier by a weighted majority
Q = np.where(P.T > 0.5, 1.0, 0.0) / K; np.fill_diagonal(Q, 0); np.fill_diagonal(Q, 1 - Q.sum(1))
Q = 0.95 * Q + 0.05 / K                                # small random jump: otherwise all mass sinks into the earliest milestone and the rest tie
ev, evec = np.linalg.eig(Q.T); pi = np.abs(np.real(evec[:, np.argmin(np.abs(ev - 1))])); pi /= pi.sum()
mc4_order = [K_lab[i] for i in np.argsort(-pi)]                                          # most visited = earliest
borda_order = [K_lab[i] for i in np.argsort(Tm @ w)]
print('Milestone (region) order, earliest -> latest:')
print('  weighted Borda:', ' '.join(borda_order)); print('  Kemeny (exact):', ' '.join(kemeny_order)); print('  MC4           :', ' '.join(mc4_order))
print(f"  rank agreement Borda vs Kemeny (Spearman) = {spearmanr([borda_order.index(k) for k in K_lab], [kemeny_order.index(k) for k in K_lab])[0]:.3f} (Spearman)")

ad.obs['t_spacecons'] = borda_w; ad.obs['t_spacecons_uniform'] = borda_u
ad.obs['rra_p_early'] = p_early; ad.obs['rra_p_late'] = p_late

# ---------- Uncertainty (method bootstrap): 95% interval per spot + local ordering confidence ----------
B = 500; boot = np.empty((B, n))
for b in range(B):
    pick = rng.integers(0, M, M); ww = w[pick] / w[pick].sum()
    boot[b] = rank01((T[:, pick] * ww).sum(1))
lo, hi = np.percentile(boot, [2.5, 97.5], 0); ad.obs['spacecons_ci_width'] = hi - lo
conf = np.array([np.mean(np.abs(2 * (boot[:, [i]] < boot[:, isp[i]]).mean(0) - 1)) for i in range(n)])   # |2·π_ij − 1| over spatial neighbours
ad.obs['spacecons_local_conf'] = conf
print(f'\nUncertainty: median 95% interval width = {np.median(hi - lo):.3f}; median local ordering confidence = {np.median(conf):.2f}')
print('Spots consistently EARLY in all methods (RRA p<0.05):', int((p_early < .05).sum()), '| consistently LATE:', int((p_late < .05).sum()))

# ---------- SPACECONS-G: consensus trajectory graph (majority rule over every method's tree on shared milestones) ----------
def majority_region(lbl):
    s = pd.Series(reg, index=ad.obs_names).groupby(pd.Series(np.asarray(lbl).astype(str), index=ad.obs_names))
    return s.agg(lambda x: x.value_counts().index[0]).to_dict()
sc.tl.paga(ad, groups='region'); Cp = ad.uns['paga']['connectivities'].toarray()
cats = list(ad.obs['region'].cat.categories)
paga_edges = {(cats[i], cats[j]): Cp[i, j] for i, j in zip(*np.triu(Cp > 0.05, 1).nonzero())}
native = {}
try: native['StaVia'] = (np.asarray(v.labels), [tuple(e) for e in v.edgelist_maxout])
except Exception: pass
try:
    pth = json.load(open(f'{OUT}/raw_stlearn_info.json')).get('psts_path', [])
    native['stLearn'] = (ad.obs['stlearn_cluster'].values, list(zip(pth[:-1], pth[1:])))
except Exception: pass
try:
    cr = sc.read_h5ad(CASCAT_RESULT); Tc = cr.uns['cascat_connectivities']
    Tc = Tc if isinstance(Tc, pd.DataFrame) else pd.DataFrame(Tc)
    native['CASCAT'] = (ad.obs['cascat_cluster'].values, [(str(Tc.index[i]), str(Tc.columns[j])) for i, j in zip(*np.nonzero(Tc.values))])
except Exception: pass
root_reg = str(ad.obs['region'].iloc[root]); trees = {}
for j, nm in enumerate(names):
    G = nx.Graph(); G.add_nodes_from(K_lab)
    for (a, b), cval in paga_edges.items(): G.add_edge(a, b, weight=cval)              # shared expression backbone (weight < 1)
    if nm in native:                                                                    # method's own graph, mapped to regions
        lbl, edges = native[nm]; mp = majority_region(lbl)
        for a, b in edges:
            ra, rb = mp.get(str(a)), mp.get(str(b))
            if ra and rb and ra != rb: G.add_edge(ra, rb, weight=G.get_edge_data(ra, rb, {'weight': 0})['weight'] + 2)
    else:                                                                               # no own graph: edges agreeing with its ordering get a bonus
        for a, b in list(G.edges()):
            G[a][b]['weight'] += 1 - abs(Tm[K_lab.index(a), j] - Tm[K_lab.index(b), j])
    if not nx.is_connected(G):
        comps = list(nx.connected_components(G))
        for c1, c2 in zip(comps[:-1], comps[1:]): G.add_edge(next(iter(c1)), next(iter(c2)), weight=1e-3)
    trees[nm] = nx.maximum_spanning_tree(G)
support = {}
for j, nm in enumerate(names):
    for a, b in trees[nm].edges(): support[frozenset((a, b))] = support.get(frozenset((a, b)), 0) + w[j]
Gs = nx.Graph(); Gs.add_nodes_from(K_lab)
for e, s in support.items(): Gs.add_edge(*tuple(e), support=s)
cons_tree = nx.maximum_spanning_tree(Gs, weight='support')                            # majority edges first; low-support edges only to connect
cl_t = pd.Series(borda_w, index=ad.obs_names).groupby(reg).mean()
cons_dir = [(a, b) if cl_t[a] <= cl_t[b] else (b, a) for a, b in cons_tree.edges()]
# Topology Disagreement Index = mean pairwise Hamming–Ipsen–Mikhailov distance between the method trees
def him(G1, G2, xi=1.0, gamma=0.08):
    A1 = nx.to_numpy_array(G1, nodelist=K_lab) > 0; A2 = nx.to_numpy_array(G2, nodelist=K_lab) > 0
    H = (A1 != A2).sum() / (K * (K - 1))
    def dens(A):
        om = np.sqrt(np.clip(np.linalg.eigvalsh(np.diag(A.sum(1)) - A.astype(float)), 0, None))[1:]
        x = np.linspace(0, om.max() + 3, 600); r = (gamma / ((x[:, None] - om[None]) ** 2 + gamma ** 2)).sum(1)
        return x, r / _trapz(r, x)
    (x1, r1), (x2, r2) = dens(A1), dens(A2); x = np.linspace(0, max(x1.max(), x2.max()), 800)
    IM = np.sqrt(_trapz((np.interp(x, x1, r1) - np.interp(x, x2, r2)) ** 2, x))
    return float(np.sqrt(H ** 2 / (1 + xi) + xi * IM ** 2 / (1 + xi)))
pairs = list(itertools.combinations(names, 2)); TDI = float(np.mean([him(trees[a], trees[b]) for a, b in pairs])) if pairs else 0.0
TDI_THRESHOLD = 0.5
topo_flag = 'topology RESOLVED' if TDI <= TDI_THRESHOLD else 'topology UNRESOLVED (methods disagree): use the ordering, treat the graph as tentative'
print(f'\nTopology Disagreement Index (mean pairwise HIM) = {TDI:.3f} -> {topo_flag}')
edge_tab = pd.DataFrame([(a, b, support.get(frozenset((a, b)), 0)) for a, b in cons_dir], columns=['from (earlier)', 'to (later)', 'support'])
print('Consensus trajectory edges (support = weighted share of methods whose tree contains the edge):')
display(edge_tab.sort_values('support', ascending=False).round(2))

# ---------- How good is the consensus? ----------
inj = ad.obs['target_score'].values
rows = [(nm, spearmanr(T[:, j], borda_w)[0], spearmanr(T[:, j], inj)[0], T[ad.obs['region'].values == ROOT_CLUSTER, j].mean()) for j, nm in enumerate(names)]
rows += [('SPACECONS (weighted)', 1.0, spearmanr(borda_w, inj)[0], borda_w[ad.obs['region'].values == ROOT_CLUSTER].mean()),
         ('Borda (uniform)', spearmanr(borda_u, borda_w)[0], spearmanr(borda_u, inj)[0], borda_u[ad.obs['region'].values == ROOT_CLUSTER].mean()),
         ('space only (1st PC of xy)', abs(spearmanr(np.linalg.svd(Z, full_matrices=False)[0][:, 0], borda_w)[0]), np.nan, np.nan)]
ev_tab = pd.DataFrame(rows, columns=['ordering', 'agreement with SPACECONS', f'corr. with {TARGET_NAME}', 'root-cluster mean']).set_index('ordering')
print('\nConsensus vs individual methods and baselines:'); display(ev_tab.round(3))
cons_sri, _ = sri(borda_w); print(f'SRI of the consensus = {cons_sri:.3f}  (1 = recoverable from position alone; tested against null models in the next step)')

# ---------- Figures ----------
img, pxy = IMG, PXY
pos = {}
for k in K_lab:
    p = pxy[reg == k]; pos[k] = p[np.argmin(((p - p.mean(0)) ** 2).sum(1))]
fig, ax = plt.subplots(1, 4, figsize=(28, 7))
for a in ax: a.imshow(img, alpha=.55); a.axis('off')
s = ax[0].scatter(*pxy.T, c=borda_w, cmap='viridis', s=9, vmin=0, vmax=1); plt.colorbar(s, ax=ax[0], fraction=.04)
ax[0].scatter(*pxy[root], c='cyan', s=220, marker='*', edgecolors='k'); ax[0].set_title('SPACECONS consensus pseudotime')
alpha = np.clip(conf, 0.05, 1); s = ax[1].scatter(*pxy.T, c=borda_w, cmap='viridis', s=9, vmin=0, vmax=1, alpha=alpha)
ax[1].set_title('confidence-masked (faded = methods disagree locally)')
s = ax[2].scatter(*pxy.T, c=hi - lo, cmap='magma', s=9); plt.colorbar(s, ax=ax[2], fraction=.04); ax[2].set_title('95% bootstrap interval width')
from matplotlib.patches import FancyArrowPatch
for a_, b_ in cons_dir:
    sup = support.get(frozenset((a_, b_)), 0)
    ax[3].add_patch(FancyArrowPatch(pos[a_], pos[b_], arrowstyle='-|>', mutation_scale=18, lw=1 + 4 * sup,
                                    color='k' if sup > 0.5 else 'grey', linestyle='-' if sup > 0.5 else '--', shrinkA=12, shrinkB=12))
ax[3].scatter([pos[k][0] for k in K_lab], [pos[k][1] for k in K_lab], c=[cl_t[k] for k in K_lab], cmap='viridis', vmin=0, vmax=1,
              s=[60 + 2 * (reg == k).sum() ** .8 for k in K_lab], edgecolors='k', zorder=3)
for k in K_lab: ax[3].text(*pos[k], k + ('*' if k == root_reg else ''), ha='center', va='center', fontsize=8, color='w', weight='bold', zorder=4)
ax[3].set_title(f'SPACECONS trajectory graph (solid = majority support)\nTDI {TDI:.2f}: {topo_flag.split(" (")[0]}')
plt.tight_layout(); plt.savefig(f'{OUT}/fig_SPACECONS.png', dpi=110, bbox_inches='tight'); plt.show()

Wt.round(4).to_csv(f'{OUT}/spacecons_weights.csv'); ev_tab.round(4).to_csv(f'{OUT}/spacecons_evaluation.csv')
edge_tab.to_csv(f'{OUT}/spacecons_graph_edges.csv', index=False)
pd.DataFrame({'barcode': ad.obs_names, 'x': xy[:, 0], 'y': xy[:, 1], 'region': reg, 'spacecons_pseudotime': borda_w,
              'ci_low': lo, 'ci_high': hi, 'local_confidence': conf, 'rra_p_early': p_early, 'rra_p_late': p_late}
             ).to_csv(f'{OUT}/spacecons_pseudotime.csv', index=False)
json.dump({'milestone_order_borda': borda_order, 'milestone_order_kemeny': kemeny_order, 'milestone_order_mc4': mc4_order,
           'TDI': TDI, 'topology': topo_flag, 'consensus_SRI': cons_sri}, open(f'{OUT}/spacecons_summary.json', 'w'), indent=1)
print('Saved spacecons_*.csv, spacecons_summary.json, fig_SPACECONS.png')


## 9. SPACECONS-M — consensus milestones and milestone graph from each method's own milestones
Each method keeps its own milestones and graph. They are compared **through the spots**: (1) per-method milestone tree, ordered by that method's pseudotime; (2) consensus milestones from the weighted co-membership matrix (their number = weighted median of the methods' own counts); (3) every method's edges transferred onto the consensus milestones, best-supported tree directed by the SPACECONS-R pseudotime; (4) consensus milestone ordering.

In [ ]:
# =====================  SPACECONS-M: consensus milestones + milestone graph from each method's OWN milestones  =====================
# Each method keeps its own milestones (its own clusters) and its own graph; the methods are compared THROUGH THE SPOTS:
#   1. per method: milestone tree (its graph, made a tree), milestones ordered by that method's pseudotime from the root
#   2. consensus milestones: co-membership matrix (weighted share of methods that put two spots in the same milestone),
#      average-linkage tree; the NUMBER of milestones = weighted median of the methods' own counts (not fixed by us)
#   3. consensus graph: every method's edges are transferred onto the consensus milestones in proportion to shared spots
#   4. consensus milestone ordering = consensus milestones sorted by the SPACECONS-R pseudotime (weighted Borda)
import json, networkx as nx
from collections import defaultdict
from scipy.cluster.hierarchy import linkage, fcluster
from scipy.spatial.distance import squareform
from scipy.integrate import quad
from scipy.linalg import eigh
from scipy.sparse.csgraph import laplacian

MCOL = {'SpaceFlow': 'm_spaceflow', 'SpaTrack': 'm_spatrack', 'StaVia': 'm_stavia', 'stLearn': 'm_stlearn', 'CASCAT': 'm_cascat'}
json.dump({k: [[str(a), str(b), float(c)] for a, b, c in v_] for k, v_ in MS_GRAPH.items()}, open(f'{OUT}/method_graphs.json', 'w'))
use = [(j, nm) for j, nm in enumerate(names) if MCOL.get(nm) in ad.obs and nm in MS_GRAPH]
print('methods with their own milestones + graph:', ', '.join(nm for _, nm in use))
n = ad.n_obs
ref = ad.obs['ref_state'].astype(object).where(ad.obs['ref_state'].notna(), '-').values

def ref_major(mask):                                     # majority reference state of a group of spots (for labelling)
    s = pd.Series(ref[mask]); s = s[s != '-']
    return s.value_counts().index[0] if len(s) else '-'

def as_tree(lab, edges, t):
    """the method's milestone graph -> one tree (max spanning tree on its edge weights); separate pieces are joined by the
    milestones closest in pseudotime (tiny weight), so every method yields one connected milestone tree."""
    ms = sorted(set(lab)); tm = pd.Series(t).groupby(lab).mean()
    G = nx.Graph(); G.add_nodes_from(ms)
    for a, b, c in edges:
        a, b = str(a), str(b)
        if a in G and b in G and a != b:
            G.add_edge(a, b, weight=max(float(c), G.get_edge_data(a, b, {'weight': 0})['weight']))
    comps = [list(c) for c in nx.connected_components(G)]
    while len(comps) > 1:
        best = min(((abs(tm[a] - tm[b]), a, b) for a in comps[0] for c in comps[1:] for b in c))
        G.add_edge(best[1], best[2], weight=1e-6, bridge=True)
        comps = [list(c) for c in nx.connected_components(G)]
    return nx.maximum_spanning_tree(G, weight='weight'), tm

trees, labs, mt = {}, {}, {}
rows = []
for j, nm in use:
    lab = ad.obs[MCOL[nm]].astype(str).values
    tree, tm = as_tree(lab, MS_GRAPH[nm], T[:, j])
    trees[nm], labs[nm], mt[nm] = tree, lab, tm
    order = tm.sort_values().index.tolist()
    rows.append({'method': nm, 'milestones': len(order), 'root milestone': lab[root],
                 'ordering (milestone[reference state])': ' → '.join(f'{m}[{ref_major(lab == m)}]' for m in order)})
MO = pd.DataFrame(rows).set_index('method')
print('\nEach method’s own milestone ordering (earliest → latest), with the majority reference state of each milestone:')
display(MO)

# ---- consensus milestones: weighted co-membership + average-linkage tree ----
wj = np.array([w[j] for j, _ in use]); wj = wj / wj.sum()
A = np.zeros((n, n), dtype=np.float32)
for (j, nm), wt in zip(use, wj):
    codes = pd.factorize(labs[nm])[0]
    A += wt * (codes[:, None] == codes[None, :])
D = 1.0 - A; np.fill_diagonal(D, 0)
Z = linkage(squareform(D, checks=False).astype(float), method='average')
# how many consensus milestones? the weighted MEDIAN of the methods' own milestone counts (the typical level of detail);
# the agreement (weighted adjusted mutual information) with the methods is printed for every candidate count as a check
from sklearn.metrics import adjusted_mutual_info_score
kc = pd.Series({nm: len(set(labs[nm])) for _, nm in use}); kw = pd.Series(wj, index=kc.index)
o = kc.sort_values().index; K_best = int(kc[o][(kw[o].cumsum() >= 0.5).values][0])
kscore = {k: sum(wt * adjusted_mutual_info_score(labs[nm], fcluster(Z, t=k, criterion='maxclust')) for (_, nm), wt in zip(use, wj))
          for k in range(max(2, kc.min()), kc.max() + 1)}
print(f'consensus milestone count = weighted median of the methods’ counts = {K_best} '
      f'(agreement with the methods, weighted AMI: {kscore[K_best]:.3f}; best possible in {kc.min()}–{kc.max()}: '
      f'{max(kscore.values()):.3f} at K = {max(kscore, key=kscore.get)})')
cm = fcluster(Z, t=K_best, criterion='maxclust')
MIN_SIZE = max(10, int(0.005 * n))                       # tiny groups are merged into the group they co-occur with most
while True:
    sizes = pd.Series(cm).value_counts(); small = sizes[sizes < MIN_SIZE].index
    if len(small) == 0 or len(sizes) <= 2: break
    k = small[0]; idx = np.where(cm == k)[0]
    others = [c for c in sizes.index if c != k]
    cm[idx] = others[int(np.argmax([A[np.ix_(idx, np.where(cm == c)[0])].mean() for c in others]))]
cmt = pd.Series(borda_w).groupby(cm).mean().sort_values()   # order consensus milestones by consensus pseudotime
rename = {c: f'C{i + 1}' for i, c in enumerate(cmt.index)}
CM = np.array([rename[c] for c in cm]); ad.obs['spacecons_milestone'] = CM
C_lab = [rename[c] for c in cmt.index]
del A, D
_counts = ', '.join(f'{nm} {MO.loc[nm, "milestones"]}' for _, nm in use)
print(f'\nConsensus milestones: {len(C_lab)} (methods have {_counts})')

# ---- consensus graph: transfer every method's tree edges onto the consensus milestones by spot overlap ----
S = np.zeros((len(C_lab), len(C_lab)))
for (j, nm), wt in zip(use, wj):
    ms = sorted(set(labs[nm]))
    O = pd.crosstab(pd.Series(labs[nm]), pd.Series(CM)).reindex(index=ms, columns=C_lab, fill_value=0)
    O = O.div(O.sum(1), axis=0).values                     # O[k, a] = share of milestone k's spots that sit in consensus milestone a
    B = nx.to_numpy_array(trees[nm], nodelist=ms, weight=None)
    Mx = O.T @ B @ O; np.fill_diagonal(Mx, 0)              # edge (k,l) votes for (a,b) with weight O[k,a]·O[l,b] (+ reverse)
    S += wt * np.minimum(Mx, 1.0)
Gs = nx.Graph(); Gs.add_nodes_from(C_lab)
for a in range(len(C_lab)):
    for b in range(a + 1, len(C_lab)):
        if S[a, b] > 0: Gs.add_edge(C_lab[a], C_lab[b], support=float(S[a, b]))
ct = pd.Series(borda_w).groupby(CM).mean()
comps = [list(c) for c in nx.connected_components(Gs)]
while len(comps) > 1:                                      # join pieces by the closest milestones in consensus pseudotime
    best = min(((abs(ct[a] - ct[b]), a, b) for a in comps[0] for c in comps[1:] for b in c))
    Gs.add_edge(best[1], best[2], support=0.0); comps = [list(c) for c in nx.connected_components(Gs)]
cons_mtree = nx.maximum_spanning_tree(Gs, weight='support')
cons_medges = sorted([((a, b) if ct[a] <= ct[b] else (b, a), Gs[a][b]['support']) for a, b in cons_mtree.edges()], key=lambda e: -e[1])
E = pd.DataFrame([(a, b, s_) for (a, b), s_ in cons_medges], columns=['from (earlier)', 'to (later)', 'support'])
comp = pd.DataFrame({'spots': pd.Series(CM).value_counts().reindex(C_lab).values,
                     'mean pseudotime': ct.reindex(C_lab).round(3).values,
                     'reference state (majority)': [ref_major(CM == c) for c in C_lab]}, index=C_lab)
print('\nConsensus milestone ordering (earliest → latest):'); display(comp)
print('Consensus milestone graph (support = weighted share of methods whose own edges connect the two milestones):'); display(E.round(2))

# ---- topology disagreement between the methods' own trees (Ipsen–Mikhailov, as in CASCAT; works across tree sizes) ----
def im_dist(G1, G2, hwhm=0.08):
    a1, a2 = nx.to_numpy_array(G1, weight=None), nx.to_numpy_array(G2, weight=None)
    w1 = np.sqrt(np.abs(eigh(laplacian(a1))[0][1:])); w2 = np.sqrt(np.abs(eigh(laplacian(a2))[0][1:]))
    n1 = (len(a1) - 1) * np.pi / 2 - np.sum(np.arctan(-w1 / hwhm)); n2 = (len(a2) - 1) * np.pi / 2 - np.sum(np.arctan(-w2 / hwhm))
    f = lambda x: (np.sum(hwhm / ((x - w1) ** 2 + hwhm ** 2)) / n1 - np.sum(hwhm / ((x - w2) ** 2 + hwhm ** 2)) / n2) ** 2
    return float(np.sqrt(quad(f, 0, np.inf, limit=100)[0]))
pairs = [(a, b) for i, a in enumerate(trees) for b in list(trees)[i + 1:]]
TDI_M = float(np.mean([im_dist(trees[a], trees[b]) for a, b in pairs])) if pairs else float('nan')
print(f'Topology disagreement between the methods’ own milestone trees (mean pairwise IM) = {TDI_M:.3f}')

# ---- figure: consensus milestones | consensus milestone graph | reference-state make-up of each consensus milestone ----
img, pxy = IMG, PXY
mpos = {c: pxy[CM == c][np.argmin(((pxy[CM == c] - pxy[CM == c].mean(0)) ** 2).sum(1))] for c in C_lab}
from matplotlib.patches import FancyArrowPatch
fig, ax = plt.subplots(1, 3, figsize=(24, 7), gridspec_kw={'width_ratios': [1, 1, 0.9]})
for a_ in ax[:2]: a_.imshow(img, alpha=.55); a_.axis('off')
cmap = plt.get_cmap('tab20')
for i, c in enumerate(C_lab):
    m_ = CM == c; ax[0].scatter(*pxy[m_].T, s=8, color=cmap(i % 20), label=c)
ax[0].legend(fontsize=8, markerscale=2, ncol=2, loc='lower left'); ax[0].set_title(f'Consensus milestones ({len(C_lab)})')
for (a_, b_), s_ in cons_medges:
    ax[1].add_patch(FancyArrowPatch(mpos[a_], mpos[b_], arrowstyle='-|>', mutation_scale=18, lw=1 + 4 * s_,
                                    color='k' if s_ >= 0.5 else 'grey', linestyle='-' if s_ >= 0.5 else '--', shrinkA=12, shrinkB=12))
sc_ = ax[1].scatter([mpos[c][0] for c in C_lab], [mpos[c][1] for c in C_lab], c=[ct[c] for c in C_lab], cmap='viridis', vmin=0, vmax=1,
                    s=[60 + 2 * (CM == c).sum() ** .8 for c in C_lab], edgecolors='k', zorder=3)
for c in C_lab: ax[1].text(*mpos[c], c, ha='center', va='center', fontsize=8, color='w', weight='bold', zorder=4)
plt.colorbar(sc_, ax=ax[1], fraction=.04, label='mean consensus pseudotime')
ax[1].set_title('Consensus milestone graph (solid = majority support)')
if HAS_REF:
    mix = pd.crosstab(pd.Series(CM, name='milestone'), pd.Series(ref, name='ref')).reindex(C_lab)
    mix = mix.drop(columns=[c for c in mix.columns if c == '-'], errors='ignore')
    (mix.div(mix.sum(1).replace(0, 1), axis=0)).plot(kind='barh', stacked=True, ax=ax[2], colormap='coolwarm', width=.8)
    ax[2].invert_yaxis(); ax[2].set_xlabel('share of reference-labelled spots'); ax[2].set_title('Reference state of each consensus milestone\n(top = earliest)')
else:
    ax[2].axis('off'); ax[2].set_title('no reference states (REF_MODE = none)')
plt.tight_layout(); plt.savefig(f'{OUT}/fig_SPACECONS_milestones.png', dpi=110, bbox_inches='tight'); plt.show()

MO.to_csv(f'{OUT}/method_milestone_orderings.csv'); comp.to_csv(f'{OUT}/spacecons_milestones.csv'); E.to_csv(f'{OUT}/spacecons_milestone_edges.csv', index=False)
pd.DataFrame({'barcode': ad.obs_names, 'spacecons_milestone': CM, 'spacecons_pseudotime': borda_w}).to_csv(f'{OUT}/spacecons_milestone_per_spot.csv', index=False)
print('Saved method_milestone_orderings.csv, spacecons_milestones.csv, spacecons_milestone_edges.csv, fig_SPACECONS_milestones.png')


## 10. Validation against the reference trajectory (CASCAT's metrics)
IM and OT distance between the predicted milestone graph and the reference chain (lower = better), Kendall τ and Spearman ρ between pseudotime and reference time, ARI / AMI of milestones vs reference labels (higher = better). Only runs when `REF_MODE` is `annotation` or `markers`.

In [ ]:
if not HAS_REF:
    print('REF_MODE = none -> no reference trajectory, validation skipped. Set REF_MODE / STATE_MAP (or MARKERS) in the settings cell to enable it.')
else:
    # =====================  Validation against the trajectory stated by stLearn, with CASCAT's metrics  =====================
    # CASCAT (utils/Metrics.py, caculate_metric) scores a method against a known trajectory with:
    #   IM  Ipsen–Mikhailov distance between the predicted milestone graph and the true milestone network   (lower = better)
    #   OT  Gromov–Wasserstein (optimal transport) distance between the same two graphs                     (lower = better)
    #   KT  Kendall τ  between predicted pseudotime and true time                                           (higher = better)
    #   SR  Spearman ρ between predicted pseudotime and true time                                           (higher = better)
    #   ARI / AMI  predicted clusters (milestones) vs true labels                                           (higher = better)
    # Here the "truth" is the trajectory stated by stLearn: true network = the chain of REF_STATES, true time = position of a
    # spot's reference state in that chain (spots without a clear reference state are left out of KT / SR), true labels = ref_label.
    # Extra column: does the method put the reference states in the stated order (mean pseudotime per state)?
    from scipy.stats import kendalltau, rankdata, spearmanr
    from sklearn.neighbors import NearestNeighbors
    from sklearn.metrics import adjusted_rand_score, adjusted_mutual_info_score
    from ot.gromov import gromov_wasserstein

    def ot_dist(G1, G2):                                      # CASCAT's OTDistance
        a1 = np.where(nx.to_numpy_array(G1) > 0, 1.0, 0); a2 = np.where(nx.to_numpy_array(G2) > 0, 1.0, 0)
        h1, h2 = np.ones(len(a1)) / len(a1), np.ones(len(a2)) / len(a2)
        _, log = gromov_wasserstein(a1, a2, h1, h2, 'square_loss', symmetric=True, log=True, verbose=False, epsilon=5e-4, max_iter=1000)
        return float(log['gw_dist'])

    REF_G = nx.Graph(); REF_G.add_nodes_from(REF_STATES); REF_G.add_edges_from(REF_EDGES)
    rt = ad.obs['ref_time'].values.astype(float); ok = np.isfinite(rt)
    rl = ad.obs['ref_label'].astype(str).values

    # NEW baseline: distance from the root THROUGH the tissue (shortest path on the 6-nearest-neighbour spot graph).
    # If a method only measures "how far from the root", it will not beat this baseline.
    from scipy.sparse.csgraph import dijkstra
    from scipy.sparse import csr_matrix
    d_, i_ = NearestNeighbors(n_neighbors=7).fit(xy).kneighbors(xy)
    Gk = csr_matrix((d_[:, 1:].ravel(), (np.repeat(np.arange(len(xy)), 6), i_[:, 1:].ravel())), shape=(len(xy), len(xy)))
    DGEO = dijkstra(Gk, directed=False, indices=root)
    DGEO[~np.isfinite(DGEO)] = np.nanmax(DGEO[np.isfinite(DGEO)])      # spots on a separate tissue piece -> farthest
    DGEO = rank01(DGEO)

    def partial_sr(pt):
        """Spearman of pseudotime with the reference time AFTER removing what distance-from-root explains (partial correlation on ranks).
        > 0: the method knows something about the trajectory beyond distance; ≈ 0: it is mostly distance."""
        a, b, c_ = (rankdata(v[ok]) for v in (pt, rt, DGEO))
        res = lambda y: y - np.polyval(np.polyfit(c_, y, 1), c_)
        return float(np.corrcoef(res(a), res(b))[0, 1])

    def state_graph(tree, lab):
        """collapse a milestone tree to the reference states: each milestone takes the majority reference state of its spots;
        milestones with no reference-labelled spots are removed and their neighbours joined; an edge between two
        different states is kept. Result: which reference states the method connects directly."""
        rs = ad.obs['ref_state'].astype(object).values; lab = np.asarray(lab).astype(str)
        maj = {}
        for m in tree.nodes():
            vv = pd.Series(rs[lab == str(m)]).dropna()
            maj[m] = vv.value_counts().index[0] if len(vv) else None
        G = tree.copy()
        for u in [m for m in list(G.nodes()) if maj[m] is None]:
            nb_ = list(G.neighbors(u)); G.add_edges_from((a, b) for i, a in enumerate(nb_) for b in nb_[i + 1:]); G.remove_node(u)
        Sg = nx.Graph(); Sg.add_nodes_from(REF_STATES)
        Sg.add_edges_from((maj[a], maj[b]) for a, b in G.edges() if maj[a] != maj[b])
        return Sg

    def edge_f1(Sg):
        E_ref = {frozenset(e) for e in REF_EDGES}; E_pred = {frozenset(e) for e in Sg.edges()}
        tp = len(E_ref & E_pred); p = tp / len(E_pred) if E_pred else 0.0; rc = tp / len(E_ref)
        return 0.0 if tp == 0 else 2 * p * rc / (p + rc)

    def score(name, pt, tree=None, lab=None):
        r = {'ordering': name}
        if tree is not None:
            r['IM (↓)'] = im_dist(REF_G, tree); r['OT (↓)'] = ot_dist(REF_G, tree)
            if lab is not None:
                Sg = state_graph(tree, lab)
                r['state-edge F1 (↑)'] = edge_f1(Sg)
                r['state edges found'] = ', '.join(sorted('–'.join(sorted(e)) for e in Sg.edges())) or '-'
        r['KT (↑)'] = kendalltau(pt[ok], rt[ok])[0]; r['SR (↑)'] = spearmanr(pt[ok], rt[ok])[0]
        r['SR beyond distance (↑)'] = partial_sr(pt)
        if lab is not None:
            r['ARI (↑)'] = adjusted_rand_score(rl, lab); r['AMI (↑)'] = adjusted_mutual_info_score(rl, lab)
            r['milestones'] = len(set(lab))
        st_mean = pd.Series(pt[ok]).groupby(rt[ok]).mean()
        r['state order KT (↑)'] = kendalltau(st_mean.index, st_mean.values)[0] if len(st_mean) > 1 else np.nan
        r['stated order recovered'] = 'yes' if np.all(np.diff(st_mean.values) > 0) else 'no'
        return r

    rows = [score(nm, T[:, j], trees.get(nm), labs.get(nm)) for j, nm in enumerate(names)]
    rows.append(score('SPACECONS (weighted)', borda_w, cons_mtree, CM))
    rows.append(score('Borda (uniform)', borda_u))
    pc1 = PCA(1).fit_transform(xy)[:, 0] if 'PCA' in globals() else np.linalg.svd(xy - xy.mean(0), full_matrices=False)[0][:, 0]
    pc1 = rank01(pc1 if pc1[root] <= np.median(pc1) else -pc1)
    rows.append(score('space only (1st PC of x, y)', pc1))
    rows.append(score('distance from root (through tissue)', DGEO))
    VAL = pd.DataFrame(rows).set_index('ordering')
    print(f'Reference trajectory (stLearn): {" → ".join(REF_STATES)} | spots with a reference state: {int(ok.sum())} of {n}')
    display(VAL.round(3))
    print('Note: IM and OT compare only the SHAPE of the milestone tree with the reference chain (state names are ignored);'
          f' with {len(REF_STATES)} reference states and many milestones they mostly measure the size difference.'
          ' state-edge F1 is label-aware: 1 = the method links exactly the stated pairs of states '
          f'({", ".join(a + "–" + b for a, b in REF_EDGES)}).')
    VAL.round(4).to_csv(f'{OUT}/validation_vs_stlearn.csv')

    st_tab = pd.DataFrame({nm: pd.Series(T[ok, j]).groupby(ad.obs['ref_state'].astype(str).values[ok]).mean() for j, nm in enumerate(names)})
    st_tab['SPACECONS'] = pd.Series(borda_w[ok]).groupby(ad.obs['ref_state'].astype(str).values[ok]).mean()
    print('Mean pseudotime of each reference state (stated order top → bottom; should increase downwards):')
    display(st_tab.reindex(REF_STATES).round(3))
    st_tab.reindex(REF_STATES).to_csv(f'{OUT}/validation_state_means.csv')
    print('Saved validation_vs_stlearn.csv and validation_state_means.csv')


## 12. Results
Everything is in `results_<RUN_NAME>/` (CSV tables, `fig_*.png`, `figures/`). The last cell packs it into one zip.

In [ ]:
# pack the results folder into one zip file
from IPython.display import FileLink, display
z = shutil.make_archive(OUT, 'zip', OUT)
print(os.path.basename(z), f'{os.path.getsize(z) / 1e6:.1f} MB')
print('results folder:', OUT)
for f in sorted(glob.glob(f'{OUT}/*.csv')): print('  ', os.path.basename(f))
display(FileLink(os.path.relpath(z)))